# VoltVision Motor Portfolio - Simulation, Pricing & Actuarial Report

**Objective.** A premium-independent cohort simulation (claims & retention driven by
experience + telematics, *not* by premium) plus a hot-swappable pricing engine
(tariff -> GLM -> GLM+Telematics) so pricing can be swapped without re-simulation.

**How to read.** Chapters 1-6 build the machinery and run one baseline book. Chapter 7 is
the long analysis report (validation -> EDA -> EV -> pricing progression). Chapter 8 is the
Monte Carlo stress test. Chapter 9 exports data and writes `report/actuarial_report.md`.

In [ ]:
import os, shutil, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gamma as gamma_dist
from scipy.stats import kstest, gamma as gamma_dist, normaltest, norm

# ensure output directories exist for figures / exports / report
os.makedirs('images', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('report/figures', exist_ok=True)

matplotlib.use('Agg')

import copy
try:
    from joblib import Parallel, delayed
    _HAS_JOBLIB = True
except ImportError:
    _HAS_JOBLIB = False
def deep_update(base, overrides):
    """Deep-copy base and recursively merge overrides (None-safe)."""
    out = copy.deepcopy(base)
    if not overrides:
        return out
    for k, v in overrides.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_update(out[k], v)
        else:
            out[k] = v
    return out


[Top 10 Vehicles in Malaysia](https://data.gov.my/dashboard/car-popularity)

[Guide to calculating insurance premium](https://bengkelbergerak.my/en/blog/kira-insurans-kereta)

[Premium calculator by Carso](https://www.carso.my/tool/car-insurance-calculator)

[Flooding risk weightage](https://www.dosm.gov.my/uploads/content-downloads/file_20220929154540.pdf)

## 1 - Configuration & Assumptions

All portfolio assumptions live in `COHORT_CONFIG` (vehicle bands, loadings, NCD table, expense / telematics loadings).

In [ ]:
# ============================================================================
# ALL MODELLING ASSUMPTIONS / CONFIG - single source of truth
#   ENGINE_CAPACITY_BANDS : tariff engine bands
#   COHORT_CONFIG         : cohort-generation assumptions
#   DRIVER_AGE_LOADING    : rating loading by driver band
#   PERIL_DIST / PERIL_BASE : claim severity model constants
# ============================================================================

ENGINE_CAPACITY_BANDS = [
    "0 to 1,400 cc / EV up to 70 kW",
    "1,401 to 1,650 cc / EV 71 - 100 kW",
    "1,651 - 2,200 cc / EV 101 - 125 kW",
    "2,201 - 3,050 cc / EV 126 - 150 kW",
    "3,051 - 4,100 cc / EV 151 - 200 kW",
    "4,101 - 4,250 cc / EV 201 - 250 kW",
    "4,251 - 4,400 cc / EV 251 - 300 kW",
    "Over 4,400 cc / EV > 300 kW",
]

COHORT_CONFIG = {
    'n': 100000,
    'coverage_pct': {
        'Comprehensive': 0.65,
        'TPFT': 0.20,
        'TPO': 0.15
    },
    'vehicle_pct': {
        'ICE': 0.90,
        'EV': 0.10
    },
    'sa_stats': {
        'ICE': {
            'lambda': 50000,
            'spread': 0.5
        },
        'EV': {
            'lambda': 80000,
            'spread': 0.5
        }
    },
    'region_pct': {
        'Peninsular Malaysia': 0.80,
        'East Malaysia (Sabah, Sawarak & Labuan)': 0.20
    },
    'generation_pct': {
        'Young Adults': 0.40,
        'Adults': 0.40,
        'Mature Adults': 0.15,
        'Seniors': 0.05
    },
    'age_bands': {
        'Young Adults': (18, 28),
        'Adults': (28, 46),
        'Mature Adults': (46, 66),
        'Seniors': (66, 76)
    },
    'gender_pct': {
        'Male': 0.55,
        'Female': 0.45
    },
    'marital_pct': {
        'Young Adults': {'Single': 0.85, 'Married': 0.15},
        'Adults': {'Single': 0.50, 'Married': 0.50},
        'Mature Adults': {'Single': 0.20, 'Married': 0.80},
        'Seniors': {'Single': 0.20, 'Married': 0.80}},
    'car_age_median': {
        'Young Adults': 2.0,
        'Adults': 3.5,
        'Mature Adults': 5.0,
        'Seniors': 5.5
    },
    'car_age_sigma': 1.5,
    'risk_pct': {
        'Peninsular Malaysia': {
            'FLOOD_RISK': [0.60, 0.40],
            'THEFT_RISK': [0.40, 0.60]
        },
        'East Malaysia (Sabah, Sawarak & Labuan)': {
            'FLOOD_RISK': [0.00, 1.00],
            'THEFT_RISK': [0.15, 0.85]
        }
    },
    'ncd_table': {
        0: 0.0000, 1: 0.2500, 2: 0.3000,
        3: 0.3833, 4: 0.4500, 5: 0.5500
    },
    'ncd_entry': {
        'years': [0, 1, 2, 3, 4, 5],
        'weights': [0.30, 0.22, 0.16, 0.13, 0.10, 0.09]
    },
    'cohort_year': 2026,
    'claim_frequency_base': -2.00,
    'ev_severity_factor': 1.20,  # explicit EV repair-cost premium on SA-bound perils (AD/Theft/Fire)

    'behavior_risk': {'lo': 0.90, 'hi': 1.30},   # claim-freq multiplier
    'telemetry': {
        'hb_shape': 2.0, 'hb_scale': 1.8, 'hb_age': -0.15, 'hb_max': 15,
        'sp_shape': 2.0, 'sp_scale': 6.0, 'sp_age': -1.0, 'sp_max': 50,
        'nd_a': 2, 'nd_b': 5, 'nd_scale': 40, 'nd_age': -1.5, 'nd_max': 50,
        'w_hb': 0.45, 'w_sp': 0.40, 'w_nd': 0.15, 'score_lo': 20, 'score_hi': 100,
    },
    'telemetric_load': 1.0000,   # extra multiplier on telematics premium (user lever)
    'expense_loading': 1.0000,  # uniform loading on GLM/telem pure premiums (LR target ~71%)
    'engine_weights': [0.25, 0.20, 0.18, 0.15, 0.10, 0.07, 0.03, 0.02],
    'seed': 42,
    'entrant_profile': {},
    # 'ev_share_by_year': {},
    # 'entrant_annual_growth': 0.00,
    'ev_share_by_year': {
        2026: 0.10, 2030: 0.12, 2035: 0.25, 2040: 0.40, 2045: 0.55
    },
    'entrant_annual_growth': 0.03,
    'entrant_ncd_zero': False,
}

MODEL_TRAIN_SEED = 20260818  # GLM/telem training cohort seed (out-of-sample vs TEST_SEED)
TEST_SEED = COHORT_CONFIG['seed']  # the priced/test book seed (=42)

COHORT_CONFIG['entrant_base_count'] = int(0.50 * COHORT_CONFIG['n'])


def interp_schedule(sched, year):
    """Linear interpolation of a {year: value} schedule (clamped at ends).
    Empty schedule -> 0.0 (no ramp configured)."""
    yrs = sorted(sched)
    if not yrs:
        return 0.0
    if year <= yrs[0]:
        return sched[yrs[0]]
    if year >= yrs[-1]:
        return sched[yrs[-1]]
    for a, b in zip(yrs, yrs[1:]):
        if a <= year <= b:
            t = (year - a) / (b - a)
            return sched[a] + t * (sched[b] - sched[a])

DRIVER_AGE_LOADING = {
    "Young Adults": 1.20,
    "Adults": 1.05,
    "Mature Adults": 1.00,
    "Seniors": 1.05,
}

PERIL_DIST = {
    'Comprehensive': {
        'AD': 0.58, 'Windscreen': 0.15, 'Theft': 0.08,
        'Fire': 0.04, 'TPPD': 0.12, 'TPBI': 0.03
    },
    'TPO': {
        'TPPD': 0.78, 'TPBI': 0.22
    },
    'TPFT': {
        'TPPD': 0.444, 'TPBI': 0.111,
        'Theft': 0.296, 'Fire': 0.148
    }
}

PERIL_BASE = {
    'TPBI':       {'shape': 0.35, 'scale': 70000, 'cap': float('inf')},
    'TPPD':       {'shape': 0.55, 'scale': 9000,  'cap': 3000000},
    'Windscreen': {'shape': 2.00, 'scale': 700,   'cap': 15000}
}



In [ ]:
# FINAL_PREMIUM_SST: BASIC x driver/car loading x (1 - NCD) x 1.1^risk flags + 8% SST
# NCD discount applies to ALL coverages (both Comprehensive and TPO).
SST_RATE = 0.08  # Changeable variable - current SST rate in Malaysia

def compute_final_premium(row, sst_rate=SST_RATE):
    """Compute final premium with rating loadings, NCD discount, risk multipliers, SST."""
    loading = total_loading(row['DRIVER_AGE_CAT'], row['CAR_AGE'])
    ncd_discount = 1 - row.get('NCD_LEVEL', 0.0)
    risk_multiplier = 1.1 ** (int(row['FLOOD_RISK']) + int(row['THEFT_RISK']))
    final = row['BASIC_PREMIUM'] * loading * ncd_discount * risk_multiplier * (1 + sst_rate)
    return round(float(final), 2)

# FINAL_PREMIUM_SST / TOTAL_LOADING applied inside generate_dataset (cell 050b235c)

# BASIC PREMIUM - Schedule of Motor Tariff 2015 (Form 5 Mathematics Ch.3)
# Graduated tariff:
#   Comprehensive = first-RM1,000 rate + PER_EXTRA x ceil((SA-1000)/1000)
#   TPFT (Third Party, Fire & Theft) = 0.75 x Comprehensive basic (Example 3)
#   TPO = flat tariff rate (no sum-assured scaling)

PER_EXTRA = {
    'Peninsular Malaysia': 26.00,
    'East Malaysia (Sabah, Sawarak & Labuan)': 20.30,
}

MOTOR_TARIFF = {
    'Peninsular Malaysia': {
        'Comprehensive': {
            '0 to 1,400 cc / EV up to 70 kW': 273.80,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 305.50,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 339.10,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 372.60,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 404.30,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 436.00,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 469.60,
            'Over 4,400 cc / EV > 300 kW': 501.30,
        },
        'TPO': {
            '0 to 1,400 cc / EV up to 70 kW': 120.60,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 135.00,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 151.20,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 167.40,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 181.80,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 196.20,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 212.40,
            'Over 4,400 cc / EV > 300 kW': 226.80,
        },
    },
    'East Malaysia (Sabah, Sawarak & Labuan)': {
        'Comprehensive': {
            '0 to 1,400 cc / EV up to 70 kW': 196.20,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 220.00,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 243.90,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 266.50,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 290.40,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 313.00,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 336.90,
            'Over 4,400 cc / EV > 300 kW': 359.50,
        },
        'TPO': {
            '0 to 1,400 cc / EV up to 70 kW': 67.50,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 75.60,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 85.20,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 93.60,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 101.70,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 110.10,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 118.20,
            'Over 4,400 cc / EV > 300 kW': 126.60,
        },
    },
}


def comprehensive_basic(row):
    first = MOTOR_TARIFF[row['REGION']]['Comprehensive'][row['ENGINE_CAPACITY']]
    units = int(np.ceil(max(0.0, (row['SUM_ASSURED'] - 1000.0) / 1000.0)))
    return first + PER_EXTRA[row['REGION']] * units


def calculate_premium(row):
    """Basic premium per Schedule of Motor Tariff 2015 (graduated)."""
    coverage = row['COVERAGE_TYPE']
    if coverage == 'Comprehensive':
        basic = comprehensive_basic(row)
    elif coverage == 'TPFT':
        basic = round(0.75 * comprehensive_basic(row) + 1e-9, 2)
    else:  # TPO - flat tariff rate
        basic = MOTOR_TARIFF[row['REGION']]['TPO'][row['ENGINE_CAPACITY']]
    return round(float(basic), 2)

# INITIAL COHORT GENERATION - single function, all assumptions in COHORT_CONFIG

def age_band(age):
    """Map an exact driver age to its rating band (band upgrades with age)."""
    if age <= 27:
        return 'Young Adults'
    if age <= 45:
        return 'Adults'
    if age <= 65:
        return 'Mature Adults'
    return 'Seniors'


def _p(weights):
    """Normalise weights to sum exactly to 1 (float-safe for np.random.choice)."""
    a = np.array(list(weights), dtype=float)
    return a / a.sum()


def generate_dataset(cfg, seed=None, n=None, cohort_year=None, polid_prefix='INIT'):
    """Generate a complete policy book from COHORT_CONFIG assumptions.

    Args:
        cfg: dict (COHORT_CONFIG) with all settings/assumptions
        seed: RNG seed (defaults to cfg['seed'])
        n: number of policies (defaults to cfg['n'])
        cohort_year: policy base year (defaults to cfg['cohort_year'])
        polid_prefix: 'INIT' for the initial book, 'ENT' for new entrants

    Returns:
        pd.DataFrame with all policy attributes + premium columns
    """
    n = int(n if n is not None else cfg['n'])
    rng = np.random.default_rng(int(seed if seed is not None else cfg['seed']))
    base_year = int(cohort_year if cohort_year is not None else cfg['cohort_year'])

    df = pd.DataFrame(index=range(n))

    # Coverage / vehicle / region
    df['COVERAGE_TYPE'] = rng.choice(
        list(cfg['coverage_pct']),
        size=n,
        p=_p(cfg['coverage_pct'].values())
    )
    df['VEHICLE_TYPE'] = rng.choice(
        list(cfg['vehicle_pct']),
        size=n,
        p=_p(cfg['vehicle_pct'].values())
    )
    df['REGION'] = rng.choice(
        list(cfg['region_pct']),
        size=n,
        p=_p(cfg['region_pct'].values())
    )

    # Sum assured: log-normal per vehicle type, rounded to RM 1,000
    sa = np.zeros(n)
    for vt, stats in cfg['sa_stats'].items():
        m = df['VEHICLE_TYPE'].values == vt
        sa[m] = rng.lognormal(
            mean=np.log(stats['lambda']),
            sigma=stats['spread'],
            size=int(m.sum())
        )
    df['SUM_ASSURED'] = np.round(sa / 1000) * 1000

    # Engine capacity: fixed hand-set mix (small cars dominant)
    df['ENGINE_CAPACITY'] = rng.choice(
        ENGINE_CAPACITY_BANDS,
        size=n,
        p=_p(cfg['engine_weights'])
    )

    # Driver profile: category by weights, age drawn from category band
    df['DRIVER_AGE_CAT'] = rng.choice(
        list(cfg['generation_pct']),
        size=n,
        p=_p(cfg['generation_pct'].values())
    )
    cats = df['DRIVER_AGE_CAT'].values
    lo = np.array([cfg['age_bands'][c][0] for c in cats])
    hi = np.array([cfg['age_bands'][c][1] for c in cats])
    df['DRIVER_AGE'] = rng.integers(lo, hi, size=n)
    df['DRIVER_GENDER'] = rng.choice(
        list(cfg['gender_pct']),
        size=n,
        p=_p(cfg['gender_pct'].values())
    )
    marital = np.empty(n, dtype=object)
    for cat, probs in cfg['marital_pct'].items():
        m = cats == cat
        marital[m] = rng.choice(
            list(probs),
            size=int(m.sum()),
            p=_p(probs.values())
        )
    df['MARITAL_STATUS'] = marital

    # Vehicle age at inception, capped 0-10
    car_age = np.zeros(n)
    for cat, med in cfg['car_age_median'].items():
        m = cats == cat
        car_age[m] = np.clip(
            np.round(med + rng.normal(0, cfg['car_age_sigma'], size=int(m.sum()))),
            0, 10
        )
    df['CAR_AGE'] = car_age.astype(int)

    # Region flood/theft risk flags
    for col in ('FLOOD_RISK', 'THEFT_RISK'):
        out = np.zeros(n, dtype=bool)
        for region, probs in cfg['risk_pct'].items():
            m = df['REGION'].values == region
            out[m] = rng.choice([True, False], size=int(m.sum()), p=_p(probs[col]))
        df[col] = out

    # NCD entry mix
    df['NCD_YEARS'] = rng.choice(
        cfg['ncd_entry']['years'],
        size=n,
        p=_p(cfg['ncd_entry']['weights'])
    )
    df['NCD_LEVEL'] = df['NCD_YEARS'].apply(lambda y: cfg['ncd_table'].get(int(min(y, 5)), 0.55))
    df['COHORT_YEAR'] = base_year

    # Telematics (teammate's feature spec): hard braking / speeding / night driving
    # -> composite telematics_score -> BEHAVIOR_RISK (drives claim frequency; mean exactly 1.10).
    # Rank-mapping (not min-max) keeps the score->risk link distortion-free: worst driver -> hi.
    tel = cfg.get('telemetry', {})
    age_z = (df['DRIVER_AGE'].values - df['DRIVER_AGE'].mean()) / df['DRIVER_AGE'].std()
    df['hard_braking_per_100km'] = np.clip(
        rng.gamma(tel.get('hb_shape', 2.0), tel.get('hb_scale', 1.8), size=n)
        + tel.get('hb_age', -0.15) * age_z, 0, tel.get('hb_max', 15))
    df['speeding_pct'] = np.clip(
        rng.gamma(tel.get('sp_shape', 2.0), tel.get('sp_scale', 6.0), size=n)
        + tel.get('sp_age', -1.0) * age_z, 0, tel.get('sp_max', 50))
    df['night_driving_pct'] = np.clip(
        rng.beta(tel.get('nd_a', 2), tel.get('nd_b', 5), size=n) * tel.get('nd_scale', 40)
        + tel.get('nd_age', -1.5) * age_z, 0, tel.get('nd_max', 50))

    def _mm(s):
        return (s - s.min()) / (s.max() - s.min())

    comp = (tel.get('w_hb', 0.45) * _mm(df['hard_braking_per_100km'])
            + tel.get('w_sp', 0.40) * _mm(df['speeding_pct'])
            + tel.get('w_nd', 0.15) * _mm(df['night_driving_pct']))
    df['telematics_score'] = np.clip(
        tel.get('score_hi', 100) - comp * (tel.get('score_hi', 100) - tel.get('score_lo', 20)),
        tel.get('score_lo', 20), tel.get('score_hi', 100))

    br = cfg.get('behavior_risk', {'lo': 0.90, 'hi': 1.30})
    _order = np.argsort(np.argsort(df['telematics_score'].values))
    _u = _order / max(int(n) - 1, 1)
    df['BEHAVIOR_RISK'] = br['hi'] - (br['hi'] - br['lo']) * _u

    # POLID: deterministic prefix-year-sequence (INIT/ENT)
    df['POLID'] = [f"{polid_prefix}{base_year}-{i + 1:06d}" for i in range(n)]

    # Premium: BASIC from tariff, TOTAL_LOADING, FINAL with SST
    df['BASIC_PREMIUM'] = df.apply(calculate_premium, axis=1)
    df['TOTAL_LOADING'] = df.apply(
        lambda r: total_loading(r['DRIVER_AGE_CAT'], r['CAR_AGE']), axis=1
    )
    df['FINAL_PREMIUM_SST'] = df.apply(compute_final_premium, axis=1)

    return df


In [ ]:
# Rating loadings: driver age category + vehicle age
# Driver loading: Youngsters highest (inexperience); experienced cohorts lower.
# Car loading: increases linearly with vehicle age (older car = higher risk).

def driver_age_loading(age_cat):
    return DRIVER_AGE_LOADING.get(age_cat, 1.00)

def car_age_loading(car_age):
    """Linear vehicle-age loading; car age capped at 10 years."""
    return 1 + 0.03 * min(int(car_age), 10)

def total_loading(age_cat, car_age):
    """Combined driver x vehicle loading applied to the premium."""
    return driver_age_loading(age_cat) * car_age_loading(car_age)

## 2 - Initial Portfolio

Generate the starting book of policies from `COHORT_CONFIG` (single deterministic call, seed 42).

In [ ]:
# Build the initial cohort (single generator call)
df = generate_dataset(COHORT_CONFIG, seed=COHORT_CONFIG['seed'])

## 3 - Risk Models - claims & retention (premium-independent)

Claim frequency (Poisson), claim severity (per-peril Gamma), and retention (driver A: experience + telematics only). None of these touch premium.

In [ ]:
# Claim Frequency Model (Poisson GLM, log-linear)
# lambda = exp(log_lambda) * coverage_multiplier
# Base exp(-2.00) ~ 0.135 claims/year - realistic Malaysian market level
# (tweak for realistic LR)

CLAIM_FREQUENCY_BASE = COHORT_CONFIG['claim_frequency_base']

def compute_claim_lambda(row):
    """Compute Poisson rate lambda via log-linear rating model."""
    log_lambda = CLAIM_FREQUENCY_BASE

    cat = row['DRIVER_AGE_CAT']
    if cat == 'Young Adults':
        log_lambda += 0.40        # young drivers: higher risk
    elif cat == 'Seniors':
        log_lambda += 0.26        # seniors: moderate increase

    # Young male interaction
    if cat == 'Young Adults' and row['DRIVER_GENDER'] == 'Male':
        log_lambda += 0.05

    # EV proxy (higher power/repair exposure)
    if row['VEHICLE_TYPE'] == 'EV':
        log_lambda += 0.05

    # Risk flags
    if row['FLOOD_RISK']:
        log_lambda += 0.20
    if row['THEFT_RISK']:
        log_lambda += 0.10

    # Behavioral risk (latent, telematics-sensed): mean 1.10 portfolio multiplier
    log_lambda += np.log(row['BEHAVIOR_RISK'])

    # Vehicle age: older cars carry higher breakdown/repair frequency
    log_lambda += 0.03 * row['CAR_AGE']

    # NCD safety credit: claim-free drivers are safer
    log_lambda -= 0.05 * row['NCD_YEARS']

    freq = np.exp(log_lambda)

    # Coverage multiplier: TPO has no own-damage exposure
    # Coverage multiplier: TPO no own-damage; TPFT fire/theft only (0.60)
    mult = 0.45 if row['COVERAGE_TYPE'] == 'TPO' else (0.60 if row['COVERAGE_TYPE'] == 'TPFT' else 1.00)
    return freq * mult


df['CLAIM_LAMBDA'] = df.apply(compute_claim_lambda, axis=1)

print('Claim frequency model (Poisson GLM) applied')
print(f"Mean lambda: {df['CLAIM_LAMBDA'].mean():.4f}")
print(f"Min lambda: {df['CLAIM_LAMBDA'].min():.4f}, "
      f"Max lambda: {df['CLAIM_LAMBDA'].max():.4f}")
print(f"TPO mean lambda: {df.loc[df['COVERAGE_TYPE']=='TPO', 'CLAIM_LAMBDA'].mean():.4f}")
print(f"Comp mean lambda: {df.loc[df['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_LAMBDA'].mean():.4f}")
print(f"TPFT mean lambda: {df.loc[df['COVERAGE_TYPE']=='TPFT', 'CLAIM_LAMBDA'].mean():.4f}")


In [ ]:
# Claim Severity Model: Per-Peril Gamma with Policy Caps
# Peril mix follows Malaysian retail product structure:
#   Comprehensive: own damage (AD/Windscreen/Theft/Fire) + third party (TPPD/TPBI)
#   TPO: third party only (TPPD/TPBI) - structurally cheaper claims

def sample_claim_peril(coverage_type):
    """Sample a claim peril from the product-specific mix."""
    mix = PERIL_DIST.get(coverage_type, PERIL_DIST['Comprehensive'])
    probs = np.array(list(mix.values()))
    return np.random.choice(list(mix.keys()), p=probs / probs.sum())


def generate_single_claim(coverage_type, sum_assured, vehicle_type=None):
    """Draw one claim amount (RM) with peril-specific Gamma + cap.
    vehicle_type='EV' applies ev_severity_factor to SA-bound perils
    (AD/Theft/Fire); mirrors the vectorized model."""
    peril = sample_claim_peril(coverage_type)
    ev_factor = COHORT_CONFIG.get('ev_severity_factor', 1.0) if vehicle_type == 'EV' else 1.0

    if peril == 'Theft':
        shape, scale = 1.10, max(8000, min(20000, sum_assured * 0.20)) * ev_factor
        cap = sum_assured
    elif peril == 'Fire':
        shape, scale = 0.90, max(7000, min(18000, sum_assured * 0.15)) * ev_factor
        cap = sum_assured
    elif peril == 'AD':
        shape, scale = 0.60, max(4500, min(12000, sum_assured * 0.10)) * ev_factor
        cap = sum_assured
    else:
        spec = PERIL_BASE[peril]
        shape, scale, cap = spec['shape'], spec['scale'], spec['cap']

    amount = np.random.gamma(shape, scale)
    return min(amount, cap), peril


def generate_claim_total(coverage_type, sum_assured, n_claims, vehicle_type=None):
    """Aggregate severity across all claims in a policy-year."""
    if n_claims <= 0:
        return 0.0, ''
    total = 0.0
    perils = []
    for _ in range(int(n_claims)):
        amt, peril = generate_single_claim(coverage_type, sum_assured, vehicle_type)
        total += amt
        perils.append(peril)
    return round(total, 2), '/'.join(perils)


print('Per-peril severity model ready:')
print('  Comprehensive perils:', list(PERIL_DIST['Comprehensive'].keys()))
print('  TPO perils:          ', list(PERIL_DIST['TPO'].keys()))
print('  TPFT perils:         ', list(PERIL_DIST['TPFT'].keys()))


In [ ]:
# Retention Model (Binomial Logit Proxy)
# Probability of renewing policy next year.
# PREMIUM_CHANGE_PCT is fed from the annual portfolio trend (see cohort-simulation).
# Note: FINAL_PREMIUM_SST now evolves yearly (loadings + NCD), but the retention
# signal remains the portfolio-level trend to keep retention behavior stable.

def compute_retention_probability(row):
    """Premium-independent retention (driver A: experience + telematics only)."""
    p = 0.82
    if row.get('CLAIM_OCCURRED', False):
        p -= 0.25
    else:
        p += 0.05
    if row['NCD_YEARS'] >= 3:
        p += 0.15
    elif row['NCD_YEARS'] >= 2:
        p += 0.08
    if row['telematics_score'] >= 80:
        p += 0.10 * (row['telematics_score'] - 80) / 20.0
    elif row['telematics_score'] >= 60:
        p += 0.03
    p -= 0.05 * (row['BEHAVIOR_RISK'] - 1.0)
    return np.clip(p, 0.10, 0.95)


## 4 - Simulation Engine

Vectorized, config-threaded cohort evolution. `simulate_cohort` ages the book, applies loadings / NCD, and emits *labels only* (no premium).

In [ ]:
# ============================================================================
# VECTORIZED SIMULATION HELPERS (NumPy) - hot-path replacement for row-wise apply
# Same models as the scalar versions (claim-model-lambda / claim-model-severity /
# sst-premium / retention-model). cfg param threads scenario overrides through.
# ============================================================================

def age_band_array(ages):
    """Vectorized band upgrade: crossing 27/45/65 moves to the next rating band."""
    return np.select(
        [ages <= 27, ages <= 45, ages <= 65],
        ['Young Adults', 'Adults', 'Mature Adults'],
        default='Seniors')


def claim_lambda_array(df, cfg=COHORT_CONFIG):
    """Vectorized Poisson frequency (log-linear GLM), same model as compute_claim_lambda."""
    cat = df['DRIVER_AGE_CAT'].values
    log_l = np.full(len(df), cfg['claim_frequency_base'])
    log_l += 0.40 * (cat == 'Young Adults')
    log_l += 0.26 * (cat == 'Seniors')
    log_l += 0.05 * ((cat == 'Young Adults') & (df['DRIVER_GENDER'].values == 'Male'))
    log_l += 0.05 * (df['VEHICLE_TYPE'].values == 'EV')
    log_l += 0.20 * df['FLOOD_RISK'].values
    log_l += 0.10 * df['THEFT_RISK'].values
    log_l += np.log(df['BEHAVIOR_RISK'].values)
    log_l += 0.03 * df['CAR_AGE'].values
    log_l -= 0.05 * df['NCD_YEARS'].values
    cov = df['COVERAGE_TYPE'].values
    mult = np.where(cov == 'TPO', 0.45, np.where(cov == 'TPFT', 0.60, 1.00))
    return np.exp(log_l) * mult


def total_loading_array(df, cfg=COHORT_CONFIG):
    """Vectorized combined driver x vehicle loading."""
    dl_map = cfg.get('driver_age_loading', DRIVER_AGE_LOADING)
    dl = df['DRIVER_AGE_CAT'].map(dl_map).fillna(1.00).values
    cl = 1 + 0.03 * np.minimum(df['CAR_AGE'].values, 10)
    return dl * cl


def final_premium_array(df, cfg=COHORT_CONFIG, sst_rate=SST_RATE):
    """Vectorized FINAL_PREMIUM_SST: BASIC x loading x (1-NCD) x 1.1^flags x (1+SST)."""
    loading = total_loading_array(df, cfg)
    ncd = 1 - df['NCD_LEVEL'].values
    risk = 1.1 ** (df['FLOOD_RISK'].values.astype(int) + df['THEFT_RISK'].values.astype(int))
    return (df['BASIC_PREMIUM'].values * loading * ncd * risk * (1 + sst_rate)).round(2)


def retention_prob_array(df):
    """Vectorized premium-independent retention (driver A)."""
    p = np.full(len(df), 0.82)
    p -= np.where(df['CLAIM_OCCURRED'].values, 0.25, -0.05)
    ncd = df['NCD_YEARS'].values
    p += np.select([ncd >= 3, ncd >= 2], [0.15, 0.08], default=0.0)
    ts = df['telematics_score'].values
    p += np.where(ts >= 80, 0.10 * (ts - 80) / 20.0,
                  np.where(ts >= 60, 0.03, 0.0))
    p -= 0.05 * (df['BEHAVIOR_RISK'].values - 1.0)
    return np.clip(p, 0.10, 0.95)


def ncd_level_array(yrs, cfg=COHORT_CONFIG):
    """Vectorized NCD discount lookup (table keyed 0-5; 6+ -> default 0.55)."""
    y = np.asarray(yrs, dtype=int)
    max_y = int(y.max()) if len(y) else 0
    lut = np.array([cfg['ncd_table'].get(min(i, 6), 0.55) for i in range(max_y + 1)])
    return lut[np.clip(y, 0, max_y)]


_PERIL_NAMES = ['AD', 'Windscreen', 'Theft', 'Fire', 'TPPD', 'TPBI']


def _draw_perils(cov, n, rng, cfg):
    """Vectorized peril draw from coverage-specific mixes (cumsum + uniform)."""
    peril_dist = cfg.get('peril_dist', PERIL_DIST)
    P = np.array([[peril_dist[c].get(p, 0.0) for p in _PERIL_NAMES] for c in cov])
    u = rng.random(n)[:, None]
    idx = np.minimum((u > np.cumsum(P, axis=1)).sum(axis=1), len(_PERIL_NAMES) - 1)
    return np.array(_PERIL_NAMES)[idx]


def _peril_params(perils, sa, cfg, ev_mult=None):
    """Vectorized Gamma shape/scale/cap per peril (caps vs sum assured).
    ev_mult: per-claim severity multiplier (e.g. EV factor) applied to
    SA-bound perils only (AD/Theft/Fire)."""
    peril_base = cfg.get('peril_base', PERIL_BASE)
    n = len(perils)
    if ev_mult is None:
        ev_mult = np.ones(n)
    shape = np.zeros(n)
    scale = np.zeros(n)
    cap = np.full(n, np.inf)
    for name in ('TPBI', 'TPPD', 'Windscreen'):
        m = perils == name
        if m.any():
            spec = peril_base[name]
            shape[m] = spec['shape']
            scale[m] = spec['scale']
            cap[m] = spec['cap']
    for name, s_lo, s_hi, s_f, sh in (('Theft', 8000, 20000, 0.20, 1.10),
                                      ('Fire', 7000, 18000, 0.15, 0.90),
                                      ('AD', 4500, 12000, 0.10, 0.60)):
        m = perils == name
        if m.any():
            shape[m] = sh
            scale[m] = np.clip(sa[m] * s_f, s_lo, s_hi) * ev_mult[m]
            cap[m] = sa[m]
    return shape, scale, cap


def simulate_claim_severity_vectorized(df, rng, cfg=COHORT_CONFIG):
    """Vectorized severity: explode by claim count -> peril -> Gamma -> cap -> aggregate."""
    n = len(df)
    amount = np.zeros(n)
    peril_out = np.empty(n, dtype=object)
    peril_out[:] = ''
    counts = df['CLAIM_COUNT'].values
    claim_mask = counts > 0
    if not claim_mask.any():
        return amount, peril_out
    idx = np.repeat(np.flatnonzero(claim_mask), counts[claim_mask])
    cov = df['COVERAGE_TYPE'].values[idx]
    sa = df['SUM_ASSURED'].values[idx]
    perils = _draw_perils(cov, len(idx), rng, cfg)
    ev_mult = np.where(
        df['VEHICLE_TYPE'].values[idx] == 'EV',
        cfg.get('ev_severity_factor', 1.0), 1.0)
    shape, scale, cap = _peril_params(perils, sa, cfg, ev_mult=ev_mult)
    amts = np.minimum(rng.gamma(shape, scale), cap)
    np.add.at(amount, idx, amts)
    s = pd.Series(perils).groupby(pd.Series(idx)).agg('/'.join)
    peril_out[np.flatnonzero(claim_mask)] = s.values
    return amount, peril_out


In [ ]:
# ============================================================================
# COHORT EVOLUTION SIMULATION (vectorized; local RNG; config-threaded)
# In-force policies age each year (DRIVER_AGE, CAR_AGE +1); claim frequency and
# final premium are recomputed annually with the new ages and NCD (one-year lag:
# year N is priced with the NCD earned through year N-1).
# ============================================================================

def _entrant_vehicle_pct(cfg, year):
    """Vehicle mix for entrants in `year`: base mix with EV share ramped by
    schedule; empty/missing schedule -> base mix unchanged (no custom EV share)."""
    base = dict(cfg['vehicle_pct'])
    sched = cfg.get('ev_share_by_year') or {}
    if not sched:
        return base
    ev = interp_schedule(sched, year)
    if ev >= 1.0:
        return {'EV': 1.0}
    others = [k for k in base if k != 'EV']
    rest = sum(base[k] for k in others)
    out = {k: (1 - ev) * base[k] / rest if rest > 0 else 0.0 for k in others}
    out['EV'] = ev
    return out


def _entrant_count(cfg, year_offset):
    """Dynamic entrant count: base * (1 + growth) ** year_offset (compounding)."""
    return max(0, int(round(cfg['entrant_base_count'] *
                            (1 + cfg['entrant_annual_growth']) ** year_offset)))


def simulate_cohort(df_initial, n_years=5, new_entrants_per_year=None,
                    seed=42, cfg=COHORT_CONFIG, verbose=True):
    """Simulate cohort evolution (vectorized hot path; per-seed local RNG).

    Args:
        df_initial: Starting cohort (Year 1)
        n_years: Number of years to simulate
        new_entrants_per_year: static entrants/year (None -> dynamic growth-based count)
        seed: Random seed for reproducibility (local Generator)
        cfg: config dict (default COHORT_CONFIG); threaded to helpers + entrants
        verbose: per-year + summary prints

    Returns:
        pd.DataFrame with all policy-year records
    """
    t0 = time.time()
    rng = np.random.default_rng(seed)
    history = []
    df_active = df_initial.copy()

    for year_offset in range(n_years):
        year = cfg['cohort_year'] + year_offset
        df_active['SIM_YEAR'] = year

        # Age in-force policies (brand-new entrants keep fresh ages).
        aging_mask = df_active['COHORT_YEAR'] < year
        if aging_mask.any():
            df_active.loc[aging_mask, 'DRIVER_AGE'] += 1
            df_active.loc[aging_mask, 'CAR_AGE'] = np.minimum(
                df_active.loc[aging_mask, 'CAR_AGE'] + 1, 10
            )
            # Band upgrades with age: crossing 27/45/65 moves to the next rating band
            df_active.loc[aging_mask, 'DRIVER_AGE_CAT'] = age_band_array(
                df_active.loc[aging_mask, 'DRIVER_AGE'].values
            )

        # Recompute frequency + loadings with current ages and NCD
        # (NCD_LEVEL here still reflects claims through the PRIOR year -> one-year lag)
        df_active['CLAIM_LAMBDA'] = claim_lambda_array(df_active, cfg)
        df_active['TOTAL_LOADING'] = total_loading_array(df_active, cfg)
        df_active['NCD_LEVEL_PRICED'] = df_active['NCD_LEVEL']

        # Simulate claims (Poisson frequency, vectorized)
        df_active['CLAIM_COUNT'] = rng.poisson(df_active['CLAIM_LAMBDA'].values)
        df_active['CLAIM_OCCURRED'] = df_active['CLAIM_COUNT'] > 0

        # Simulate severity (per-peril Gamma, vectorized, aggregated per policy-year)
        amounts, perils = simulate_claim_severity_vectorized(df_active, rng, cfg)
        df_active['CLAIM_AMOUNT'] = amounts
        df_active['CLAIM_PERIL'] = perils

        # Retention is premium-independent (driver A: experience + telematics)

        # Retention + renewals (vectorized)
        df_active['RENEWAL_PROB'] = retention_prob_array(df_active)
        df_active['RENEWED'] = rng.random(len(df_active)) < df_active['RENEWAL_PROB'].values

        # Update NCD based on claims
        df_active.loc[~df_active['CLAIM_OCCURRED'], 'NCD_YEARS'] += 1
        df_active.loc[df_active['CLAIM_OCCURRED'], 'NCD_YEARS'] = 0
        df_active['NCD_LEVEL'] = ncd_level_array(df_active['NCD_YEARS'].values, cfg)

        # Record full year state (including lapsers) for retention analysis
        cols_to_keep = ['POLID', 'COVERAGE_TYPE', 'SUM_ASSURED', 'REGION',
                        'VEHICLE_TYPE', 'DRIVER_AGE_CAT', 'DRIVER_AGE',
                        'CAR_AGE', 'DRIVER_GENDER', 'FLOOD_RISK', 'THEFT_RISK',
                        'BASIC_PREMIUM', 'TOTAL_LOADING',
                        'NCD_LEVEL_PRICED', 'NCD_LEVEL',
                        'NCD_YEARS', 'CLAIM_LAMBDA',
                        'SIM_YEAR', 'CLAIM_COUNT', 'CLAIM_OCCURRED', 'CLAIM_AMOUNT',
                        'CLAIM_PERIL',
                        'RENEWAL_PROB', 'RENEWED', 'COHORT_YEAR',
                        'BEHAVIOR_RISK', 'telematics_score']
        history.append(df_active[cols_to_keep].copy())

        if verbose:
            print(f"Year {year}: {len(df_active)} active policies, "
                  f"claims: {df_active['CLAIM_COUNT'].sum()}, "
                  f"freq: {df_active['CLAIM_OCCURRED'].mean():.1%}, "
                  f"avg NCD priced: {df_active['NCD_LEVEL_PRICED'].mean():.2%}, "
                  f"retention: {df_active['RENEWED'].mean():.1%}")

        # New entrants: dynamic profile (EV share ramps by year) + dynamic count
        # (base * (1 + growth)**year_offset) unless a static override is passed.
        if year_offset < n_years - 1:
            n_ent = (new_entrants_per_year if new_entrants_per_year is not None
                     else _entrant_count(cfg, year_offset))
            if n_ent > 0:
                ecfg = copy.deepcopy(cfg)
                for k, v in cfg.get('entrant_profile', {}).items():
                    ecfg[k] = v
                ecfg['vehicle_pct'] = _entrant_vehicle_pct(cfg, year + 1)
                new_cohort = generate_dataset(
                    ecfg, seed=seed + year_offset,
                    n=n_ent, cohort_year=year + 1,
                    polid_prefix='ENT'
                )
                if cfg.get('entrant_ncd_zero', False):
                    new_cohort['NCD_YEARS'] = 0
                    new_cohort['NCD_LEVEL'] = cfg['ncd_table'].get(0, 0.55)
                df_active = pd.concat(
                    [df_active[df_active['RENEWED']], new_cohort],
                    ignore_index=True
                )
            else:
                df_active = df_active[df_active['RENEWED']].copy()
        else:
            df_active = df_active[df_active['RENEWED']].copy()

    cohort_results = pd.concat(history, ignore_index=True)
    if verbose:
        print(f"\nSimulation complete. Total records: {len(cohort_results)}")
        print(f"Year range: {cohort_results['SIM_YEAR'].min()} - {cohort_results['SIM_YEAR'].max()}")
        print(f"Simulation wall time: {time.time() - t0:.1f}s")
    return cohort_results


## 5 - Pricing Engine (hot-swappable)

`price_book(book, method)` places premium post-simulation via the registry: `tariff` (rule-based), `glm` (traditional factors), `telem` (adds `telematics_score`). `compare_pricing` summarises all three.

In [ ]:
# ============================================================================
# PREMIUM PRICING ENGINE (hot-swappable) - placed AFTER simulation, decoupled
# from simulate_cohort (now premium-independent / labels only).
# Registry: 'tariff' (rule-based final_premium_array), 'glm', 'telem'.
# ============================================================================

from sklearn.linear_model import PoissonRegressor
from scipy.stats import spearmanr

_PRICING_FEATURES = {
    'glm': ['DRIVER_AGE', 'CAR_AGE', 'NCD_LEVEL_PRICED', 'VEHICLE_TYPE',
            'COVERAGE_TYPE', 'FLOOD_RISK', 'THEFT_RISK', 'REGION'],
    'telem': ['DRIVER_AGE', 'CAR_AGE', 'NCD_LEVEL_PRICED', 'VEHICLE_TYPE',
              'COVERAGE_TYPE', 'FLOOD_RISK', 'THEFT_RISK', 'REGION',
              'telematics_score'],
}


def _pricing_features(method):
    if method in _PRICING_FEATURES:
        return list(_PRICING_FEATURES[method])
    raise ValueError('unknown pricing method: ' + str(method))


def _encode_features(df, features):
    _CAT = ['VEHICLE_TYPE', 'COVERAGE_TYPE', 'REGION']
    X = df[features].copy()
    for col in _CAT:
        if col in features:
            X[col] = X[col].astype('category').cat.codes
    return X


def _require_simulated(book):
    """Fail fast if the book was not produced by simulate_cohort().
    Training/pricing must always run on a simulated book, never on raw inputs.
    """
    required = ['SIM_YEAR', 'COHORT_YEAR', 'POLID', 'CLAIM_COUNT', 'CLAIM_AMOUNT']
    missing = [c for c in required if c not in book.columns]
    if missing:
        raise ValueError(
            'train_pricing/price_book requires a SIMULATED book, but these columns '
            f'are missing: {missing}. Run simulate_cohort() before pricing.')
    if book['CLAIM_COUNT'].isnull().any() or book['CLAIM_AMOUNT'].isnull().any():
        raise ValueError('book has null claim fields - run simulate_cohort() first.')


def train_pricing(book, method='tariff', cfg=COHORT_CONFIG, train_seed=None):
    """Fit a frequency model on entrant policy-years (no claim-history endogeneity).

    Target = CLAIM_COUNT; severity priced separately from avg severity by coverage.
    """
    _require_simulated(book)
    features = _pricing_features(method)
    if train_seed is not None:
        _tr_book = simulate_cohort(generate_dataset(cfg, seed=train_seed),
                                   n_years=5, new_entrants_per_year=(0.5*cfg['n']), seed=train_seed, cfg=cfg, verbose=False)
    else:
        _tr_book = book
    tr_src = _tr_book[_tr_book['COHORT_YEAR'] == _tr_book['SIM_YEAR']]
    pids = tr_src['POLID'].unique()
    train_pids = set(np.random.default_rng(7).choice(
        pids, int(len(pids) * 0.6), replace=False))
    tr = tr_src[tr_src['POLID'].isin(train_pids)]
    model = PoissonRegressor(alpha=1e-3, max_iter=1000).fit(
        _encode_features(tr, features), tr['CLAIM_COUNT'])
    avg_sev_by_cov = _tr_book.groupby('COVERAGE_TYPE').apply(
        lambda d: d['CLAIM_AMOUNT'].sum() / d['CLAIM_COUNT'].sum(),
        include_groups=False).to_dict()
    return {'features': features, 'model': model,
            'avg_sev_by_cov': avg_sev_by_cov,
            'expense_loading': cfg.get('expense_loading', 1.40),
            'telemetric_load': cfg.get('telemetric_load', 1.0)}


def price_book(book, method='tariff', cfg=COHORT_CONFIG, train_seed=None):
    """Return a copy of `book` with FINAL_PREMIUM_SST computed by `method`."""
    out = book.copy()
    if method == 'tariff':
        out['FINAL_PREMIUM_SST'] = final_premium_array(out, cfg)
        return out
    pricer = train_pricing(book, method, cfg, train_seed=train_seed)
    X = _encode_features(out, pricer['features'])
    pf = pricer['model'].predict(X)
    sev = out['COVERAGE_TYPE'].map(pricer['avg_sev_by_cov']).astype(float).values
    prem = (pf * sev * pricer['expense_loading'] * pricer['telemetric_load'])
    prem = prem * (1.1 ** out['FLOOD_RISK'].values.astype(int))
    prem = prem * (1.1 ** out['THEFT_RISK'].values.astype(int))
    out['FINAL_PREMIUM_SST'] = prem.round(2)
    return out


def _retained_lr(res, decline_pct=0.15):
    p = res.groupby('POLID').agg(claims=('CLAIM_AMOUNT', 'sum'),
                                 prem=('FINAL_PREMIUM_SST', 'sum')).reset_index()
    k = int(np.floor(len(p) * decline_pct))
    p = p.sort_values('prem', ascending=False).iloc[k:]
    return p['claims'].sum() / p['prem'].sum() * 100


def _tier_spread(res):
    d = res.copy()
    d['_bin'] = pd.cut(d['telematics_score'], bins=[0, 50, 70, 85, 100],
                       labels=['<50', '50-70', '70-85', '85-100'])
    g = d.groupby('_bin', observed=False).apply(
        lambda x: x['CLAIM_AMOUNT'].sum() / x['FINAL_PREMIUM_SST'].sum() * 100)
    return g.max() - g.min()


def compare_pricing(book, methods=('tariff', 'glm', 'telem'), cfg=COHORT_CONFIG, train_seed=None):
    rows = []
    for m in methods:
        b = price_book(book, m, cfg, train_seed=train_seed)
        rows.append({
            'model': m,
            'overall_LR(%)': round(b['CLAIM_AMOUNT'].sum() / b['FINAL_PREMIUM_SST'].sum() * 100, 2),
            'retained_LR(%)': round(_retained_lr(b, 0.15), 2),
            'prem_count_rho': round(spearmanr(b['FINAL_PREMIUM_SST'], b['CLAIM_COUNT']).correlation, 4),
            'tier_spread(pp)': round(_tier_spread(b), 1),
            'avg_premium': round(b['FINAL_PREMIUM_SST'].mean(), 2),
        })
    return pd.DataFrame(rows)


## 6 - Baseline Run

One trajectory, priced with the TARIFF regime as the reference book. All three regimes (tariff / GLM / GLM+Telematics) are compared side-by-side; the GLM/telem models are trained out-of-sample on seed {MODEL_TRAIN_SEED} and tested on the book seed {TEST_SEED}.

In [ ]:
# Run simulation (single trajectory, premium-independent / labels only)
cohort_results = simulate_cohort(
    df, n_years=5, new_entrants_per_year=None)
cohort_results = price_book(cohort_results, 'tariff', COHORT_CONFIG)

# --- Calibrate expense_loading so GLM/telem land in the 65-75% LR band ---
def _base_lr(test_book, method, train_seed):
    b = price_book(test_book, method, COHORT_CONFIG, train_seed=train_seed)
    return b['CLAIM_AMOUNT'].sum() / b['FINAL_PREMIUM_SST'].sum()

_glm_base = _base_lr(cohort_results, 'glm', MODEL_TRAIN_SEED)
_el = _glm_base / 0.70
_el = min(max(_el, 0.85), 1.60)
COHORT_CONFIG['expense_loading'] = round(float(_el), 4)
print(f'CALIBRATION: GLM base LR (el=1.0) = {_glm_base*100:.2f}% -> '
      f'expense_loading set to {COHORT_CONFIG["expense_loading"]:.4f} '
      f'(target ~70% GLM/telem LR; tariff unaffected)')

# --- Three regimes, side-by-side (GLM/telem trained on MODEL_TRAIN_SEED, tested on TEST_SEED) ---
BOOKS = {
    'tariff': cohort_results,
    'glm':    price_book(cohort_results, 'glm', COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED),
    'telem':  price_book(cohort_results, 'telem', COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED),
}
print('Three regimes priced (side-by-side):', list(BOOKS.keys()))


## 7 - Analysis Report

Validation, exploratory analysis, EV market analysis, and the tariff -> GLM -> telem pricing progression - run consecutively as one continuous report.

In [ ]:
# Test: Sample Claim Generation
# Demonstrate claim modeling on sample policies

def generate_sample_claims(df, n=5):
    """Generate sample claims for testing the model."""
    samples = df.sample(n=min(n, len(df)), random_state=42)

    print('=== Sample Claim Generation ===\n')

    for idx, row in samples.iterrows():
        lamb = row['CLAIM_LAMBDA']
        n_claims = np.random.poisson(lamb)

        print(f"Policy: {row['POLID'][:12]}...")
        print(f"  Coverage: {row['COVERAGE_TYPE']}, Vehicle: {row['VEHICLE_TYPE']}")
        print(f"  Sum Assured: RM{row['SUM_ASSURED']:,.0f}")
        print(f"  Flood Risk: {row['FLOOD_RISK']}, Theft Risk: {row['THEFT_RISK']}")
        print(f"  NCD Years: {row['NCD_YEARS']} ({row['NCD_LEVEL']:.1%})")
        print(f"  Claim Intensity (lambda): {lamb:.3f}")

        if n_claims > 0:
            total_claim, perils = generate_claim_total(
                row['COVERAGE_TYPE'], row['SUM_ASSURED'], n_claims
            )
            print(f"  Perils: {perils}")
            print(f"  TOTAL CLAIM: RM{total_claim:,.2f}")
        else:
            print('  No claims this year')

        print(f"  Basic Premium: RM{row['BASIC_PREMIUM']:,.2f}")
        print(f"  Final Premium (w/ SST): RM{row['FINAL_PREMIUM_SST']:,.2f}")
        print()


# Run sample generation
generate_sample_claims(df, n=1)


In [ ]:

# ============================================================
# Statistical Validation (Part A: Core Tests)
# ============================================================

results = cohort_results.copy()
passed = []
failed = []

def check(name, cond, detail=''):
    if cond:
        passed.append(name)
        print(f"[PASS] {name}")
    else:
        failed.append(name)
        print(f"[FAIL] {name} {detail}")

# Test 1: Overall claim frequency in plausible band (10%-20%)
overall_freq = results['CLAIM_OCCURRED'].mean()
check('1. Overall claim frequency within 10%-20%',
      0.10 <= overall_freq <= 0.20,
      f"(actual {overall_freq:.1%})")

# Test 2: TPO expected claim cost per policy-year < Comprehensive
# (frequency x mean severity - TPO has no own-damage exposure)
comp_freq_t = results.loc[results['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_OCCURRED'].mean()
tpo_freq_t = results.loc[results['COVERAGE_TYPE']=='TPO', 'CLAIM_OCCURRED'].mean()
comp_sev = results.loc[(results['COVERAGE_TYPE']=='Comprehensive') &
                       (results['CLAIM_AMOUNT']>0), 'CLAIM_AMOUNT']
tpo_sev = results.loc[(results['COVERAGE_TYPE']=='TPO') &
                      (results['CLAIM_AMOUNT']>0), 'CLAIM_AMOUNT']
comp_mean = comp_sev.mean() if len(comp_sev) else 0
tpo_mean = tpo_sev.mean() if len(tpo_sev) else 0
comp_cost = comp_freq_t * comp_mean
tpo_cost = tpo_freq_t * tpo_mean
check('2. TPO expected claim cost/policy-year < Comprehensive',
      tpo_cost < comp_cost,
      f"(Comp RM{comp_cost:,.0f} vs TPO RM{tpo_cost:,.0f})")

# Test 3: Per-peril means (report; caps enforced at draw time)
peril_means = (results[results['CLAIM_AMOUNT']>0]
               .assign(peril_first=lambda d: d['CLAIM_PERIL'].str.split('/').str[0])
               .groupby('peril_first')['CLAIM_AMOUNT'].mean())
print('\n  Per-peril mean severity:')
for peril, m in peril_means.sort_values(ascending=False).items():
    print(f"    {peril:10s} RM{m:,.0f}  (n={len(results[results['CLAIM_PERIL'].str.contains(peril)])})")
print('  (caps enforced at draw time by construction)')

# Test 4: Loss ratio (incurred / earned premium) - report only
earned = results['FINAL_PREMIUM_SST'].sum()
incurred = results['CLAIM_AMOUNT'].sum()
loss_ratio = incurred / earned if earned > 0 else float('nan')
print(f"\n  Loss ratio: {loss_ratio:.1%} (earned RM{earned:,.0f}, incurred RM{incurred:,.0f})")
if not (0.40 <= loss_ratio <= 0.80):
    print(f"  [WARN] Loss ratio outside 40%-80% band - inspect premium adequacy")
else:
    print(f"  [OK]   Loss ratio within 40%-80% band")

# Test 5: NCD mechanics - claim resets to 0, claim-free increments
reset_ok = (results.loc[results['CLAIM_OCCURRED'], 'NCD_YEARS'] == 0).mean()
inc_ok = (results.loc[~results['CLAIM_OCCURRED'], 'NCD_YEARS'] >= 1).mean()
check('5a. Claims reset NCD_YEARS to 0', reset_ok >= 0.99,
      f"(reset rate {reset_ok:.1%})")
check('5b. Claim-free years increment NCD', inc_ok >= 0.99,
      f"(increment rate {inc_ok:.1%})")

# Test 6: TPO frequency < Comprehensive frequency
comp_freq = results.loc[results['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_OCCURRED'].mean()
tpo_freq = results.loc[results['COVERAGE_TYPE']=='TPO', 'CLAIM_OCCURRED'].mean()
check('6. TPO claim frequency < Comprehensive',
      tpo_freq < comp_freq,
      f"(Comp {comp_freq:.1%} vs TPO {tpo_freq:.1%})")

# Test 7: Data integrity - no nulls/negatives in key columns
key_cols = ['CLAIM_COUNT', 'CLAIM_AMOUNT', 'CLAIM_LAMBDA', 'FINAL_PREMIUM_SST',
            'NCD_LEVEL', 'NCD_YEARS', 'RENEWED', 'CAR_AGE', 'TOTAL_LOADING',
            'NCD_LEVEL_PRICED']
null_bad = results[key_cols].isnull().sum().sum()
neg_bad = (results['CLAIM_AMOUNT'] < 0).sum() + (results['FINAL_PREMIUM_SST'] <= 0).sum()
check('7. No nulls in key columns', null_bad == 0, f"(nulls: {null_bad})")
check('7b. No negative/zero premium or negative claims', neg_bad == 0,
      f"(bad: {neg_bad})")

# Test 8b: Ageing works - in-force DRIVER_AGE/CAR_AGE increase across years
age_by_year = results.groupby('SIM_YEAR')[['DRIVER_AGE', 'CAR_AGE']].mean()
age_growth = age_by_year['DRIVER_AGE'].iloc[-1] > age_by_year['DRIVER_AGE'].iloc[0]
car_growth = age_by_year['CAR_AGE'].iloc[-1] > age_by_year['CAR_AGE'].iloc[0]
check('8b. Mean DRIVER_AGE rises across years', age_growth,
      f"({age_by_year['DRIVER_AGE'].iloc[0]:.1f} -> {age_by_year['DRIVER_AGE'].iloc[-1]:.1f})")
check('8c. Mean CAR_AGE rises across years', car_growth,
      f"({age_by_year['CAR_AGE'].iloc[0]:.1f} -> {age_by_year['CAR_AGE'].iloc[-1]:.1f})")

# Test 9: CAR_AGE plausibility
# Fleet age at inception (year-1 policies), not the aged 20-yr book
car_age_med = results.loc[results['SIM_YEAR'] == COHORT_CONFIG['cohort_year'], 'CAR_AGE'].median()
check('9. CAR_AGE median within 3-4 years', 3.0 <= car_age_med <= 4.0,
      f"(median {car_age_med:.1f})")
check('9b. CAR_AGE within [0,10]', results['CAR_AGE'].between(0, 10).all())
genz_car = results.loc[results['DRIVER_AGE_CAT'] == 'Young Adults', 'CAR_AGE'].mean()
other_car = results.loc[results['DRIVER_AGE_CAT'] != 'Young Adults', 'CAR_AGE'].mean()
check('9c. Young Adults drive newer cars than other bands', genz_car < other_car,
      f"(Young Adults {genz_car:.1f} vs others {other_car:.1f})")

# Test 10: Premium evolves across years for in-force policies
years_per_polid = results.groupby('POLID')['SIM_YEAR'].nunique()
multi_year = results[results['POLID'].isin(
    years_per_polid[years_per_polid >= 3].index
)]
prem_levels = multi_year.groupby('POLID')['FINAL_PREMIUM_SST'].nunique()
prem_varies = (prem_levels > 1).mean()
check('10. Premium varies across years for multi-year policies',
      prem_varies >= 0.9, f"({prem_varies:.1%} of policies vary)")

# Test 11: Dynamic entrants - EV adoption ramp + growth-based count
_sched = COHORT_CONFIG.get('ev_share_by_year') or {}
_has_ramp = len(_sched) > 1 and (max(_sched.values()) - min(_sched.values())) > 1e-9
ent = results[results['POLID'].str.startswith('ENT')]
if not _has_ramp:
    print('[SKIP] Test 11 (EV share constant or unset)')
elif len(ent) and len(ent['SIM_YEAR'].unique()) > 1:
    ev_by_year = ent.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())
    check('11a. Entrant EV share rises across years',
          ev_by_year.iloc[-1] > ev_by_year.iloc[0],
          f"({ev_by_year.iloc[0]:.1%} -> {ev_by_year.iloc[-1]:.1%})")
    check('11b. Entrant EV share <= 0.95', ev_by_year.max() <= 0.95,
          f"(max {ev_by_year.max():.1%})")
    all_ev = results.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())
    check('11c. Portfolio EV share rises across years',
          all_ev.iloc[-1] > all_ev.iloc[0],
          f"({all_ev.iloc[0]:.1%} -> {all_ev.iloc[-1]:.1%})")
    if COHORT_CONFIG['entrant_annual_growth'] > 0:
        ent_cnt = ent.groupby('SIM_YEAR')['POLID'].count()
        check('11d. Entrant count grows with annual growth',
              ent_cnt.iloc[-1] > ent_cnt.iloc[0],
              f"({ent_cnt.iloc[0]:,} -> {ent_cnt.iloc[-1]:,})")
    else:
        print('[SKIP] 11d. Entrant count growth (annual growth = 0)')
else:
    print('[SKIP] Test 11 (no multi-year entrant data)')

# Test 12: Explicit EV severity premium (ev_severity_factor) on SA-bound perils
_claims = results[results['CLAIM_OCCURRED']].copy()
_sb = _claims[_claims['CLAIM_PERIL'].isin(['AD', 'Theft', 'Fire'])]
_tp = _claims[_claims['CLAIM_PERIL'].isin(['TPBI', 'TPPD', 'Windscreen'])]
if len(_sb) and len(_tp):
    _sb_ev = _sb.loc[_sb['VEHICLE_TYPE'] == 'EV', 'CLAIM_AMOUNT'].mean()
    _sb_ice = _sb.loc[_sb['VEHICLE_TYPE'] == 'ICE', 'CLAIM_AMOUNT'].mean()
    check('12a. EV severity > ICE on SA-bound perils (AD/Theft/Fire)',
          _sb_ev > _sb_ice, f"(EV RM{_sb_ev:,.0f} vs ICE RM{_sb_ice:,.0f})")
    _tp_ev = _tp.loc[_tp['VEHICLE_TYPE'] == 'EV', 'CLAIM_AMOUNT'].mean()
    _tp_ice = _tp.loc[_tp['VEHICLE_TYPE'] == 'ICE', 'CLAIM_AMOUNT'].mean()
    _ratio = _tp_ev / _tp_ice if _tp_ice > 0 else float('nan')
    check('12b. TP perils severity EV ~= ICE (factor scope-limited)',
          pd.notna(_ratio) and 0.8 <= _ratio <= 1.25,
          f"(EV RM{_tp_ev:,.0f} vs ICE RM{_tp_ice:,.0f}, ratio {_ratio:.2f})")
else:
    print('[SKIP] Test 12 (no claims to compare)')

print(f"\n===== VALIDATION RESULT: {len(passed)} passed, {len(failed)} failed =====")


In [ ]:

# ============================================================
# Statistical Validation (Part B) + EDA Enrichment
# ============================================================

# Test 8: Reproducibility - re-running with same seed gives identical claims
np.random.seed(999)
run_a = simulate_cohort(df, n_years=3, seed=7)

np.random.seed(999)
run_b = simulate_cohort(df, n_years=3, seed=7)

same_claims = (run_a['CLAIM_COUNT'].values == run_b['CLAIM_COUNT'].values).all()
same_amounts = np.allclose(run_a['CLAIM_AMOUNT'].values, run_b['CLAIM_AMOUNT'].values)

if same_claims and same_amounts:
    print("[PASS] 8. Reproducibility: same seed -> identical claims")
else:
    print("[FAIL] 8. Reproducibility: seeded runs differ")

# ---- EDA enrichment ----
res = cohort_results.copy()

# E1: Loss ratio by coverage type (n/a if no earned premium)
def lr_ratio(d):
    earned = d['FINAL_PREMIUM_SST'].sum()
    if earned <= 0:
        return 'n/a (no earned premium)'
    return f"{d['CLAIM_AMOUNT'].sum() / earned:.1%}"
lr_by_cov = res.groupby('COVERAGE_TYPE').apply(lr_ratio)
print('\nE1. Loss ratio by coverage type (TPO underpriced - see FINDING 1):')
print(lr_by_cov.to_string())

# E2: Peril mix across all claims
peril_counts = res[res['CLAIM_AMOUNT'] > 0]['CLAIM_PERIL'].str.split('/').explode().value_counts()
print('\nE2. Peril mix:')
print(peril_counts.to_string())

# E3: Claim frequency by age category
freq_by_age = res.groupby('DRIVER_AGE_CAT')['CLAIM_OCCURRED'].mean().sort_values(ascending=False)
print('\nE3. Claim frequency by age category:')
print(freq_by_age.map(lambda x: f"{x:.1%}").to_string())

# E4: Retention by claim status
ret_by_claim = res.groupby('CLAIM_OCCURRED')['RENEWED'].mean()
print('\nE4. Retention by claim status:')
print(ret_by_claim.map(lambda x: f"{x:.1%}").to_string())


In [ ]:
# ============================================================================
# OVERTHINKER-STYLE VALIDATION (Part A + Enhanced Tests + Correlation)
# Adapted from reference overthinker_gen_data.ipynb Section 6
# ============================================================================

final_dataset = cohort_results.copy()

print("="*70)
print("DATASET VALIDATION")
print("="*70)

validation_results = []

# 1. Premium vs Sum Insured Correlation (Comprehensive; TPO premium is SA-free by design)
comp_sub = final_dataset[final_dataset['COVERAGE_TYPE'] == 'Comprehensive']
corr_comp = comp_sub['FINAL_PREMIUM_SST'].corr(comp_sub['SUM_ASSURED'])
validation_results.append({
    'Test': 'Premium vs Sum Insured Correlation (Comp)',
    'Value': f"{corr_comp:.3f}",
    'Expected': '> 0.5',
    'Pass': corr_comp > 0.5
})

# 2. Young Driver Premium Loading
young_avg = final_dataset[final_dataset['DRIVER_AGE'] < 25]['FINAL_PREMIUM_SST'].mean()
mature_avg = final_dataset[final_dataset['DRIVER_AGE'].between(30, 50)]['FINAL_PREMIUM_SST'].mean()
loading = young_avg / mature_avg
validation_results.append({
    'Test': 'Young Driver Premium Loading',
    'Value': f"{loading:.2f}x",
    'Expected': '> 1.05x (driver loading partly offset by newer cars)',
    'Pass': loading > 1.05
})

# 3. Claim Frequency Range
claim_freq = (final_dataset['CLAIM_COUNT'] > 0).mean()
validation_results.append({
    'Test': 'Overall Claim Frequency',
    'Value': f"{claim_freq*100:.2f}%",
    'Expected': '10-20%',
    'Pass': 0.10 <= claim_freq <= 0.20
})

# 4. NCD Progression
ncd_2026 = final_dataset[final_dataset['SIM_YEAR'] == 2026]['NCD_LEVEL_PRICED'].mean()
ncd_2030 = final_dataset[final_dataset['SIM_YEAR'] == 2030]['NCD_LEVEL_PRICED'].mean()
validation_results.append({
    'Test': 'NCD Progression (2026 < 2030)',
    'Value': f"{ncd_2026*100:.1f}% to {ncd_2030*100:.1f}%",
    'Expected': 'Increasing',
    'Pass': ncd_2030 > ncd_2026
})

# 5. Loss Ratio Range
loss_ratio = final_dataset['CLAIM_AMOUNT'].sum() / final_dataset['FINAL_PREMIUM_SST'].sum()
validation_results.append({
    'Test': 'Overall Loss Ratio',
    'Value': f"{loss_ratio:.2%}",
    'Expected': '50-80%',
    'Pass': 0.50 <= loss_ratio <= 0.80
})

# 6. Region premium difference (report only - rate file dependent)
east_avg = final_dataset[final_dataset['REGION'].str.contains('East')]['FINAL_PREMIUM_SST'].mean()
pen_avg = final_dataset[final_dataset['REGION'].str.contains('Peninsular')]['FINAL_PREMIUM_SST'].mean()
region_ratio = east_avg / pen_avg if pen_avg > 0 else float('nan')
print(f"  Region premium ratio (East/Peninsular): {region_ratio:.2f}x (report only)")

# 7. Comprehensive vs TPO Premium
comp_avg = final_dataset[final_dataset['COVERAGE_TYPE'] == 'Comprehensive']['FINAL_PREMIUM_SST'].mean()
tpo_avg = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPO']['FINAL_PREMIUM_SST'].mean()
comp_ratio = comp_avg / tpo_avg if tpo_avg > 0 else float('nan')
validation_results.append({
    'Test': 'Comprehensive Premium > TPO',
    'Value': f"{comp_ratio:.2f}x",
    'Expected': '> 1.8x',
    'Pass': comp_ratio > 1.8
})

# 7b. TPFT premium between TPO and Comprehensive
tpft_avg = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPFT']['FINAL_PREMIUM_SST'].mean()
validation_results.append({
    'Test': 'TPFT Premium between TPO & Comp',
    'Value': f"RM{tpo_avg:,.0f} < RM{tpft_avg:,.0f} < RM{comp_avg:,.0f}",
    'Expected': 'TPO < TPFT < Comprehensive',
    'Pass': tpo_avg < tpft_avg < comp_avg
})

# 8. NCD Discount (controlled: same policy priced at NCD=0 vs actual)
def premium_at_zero_ncd(row):
    r = row.copy()
    r['NCD_LEVEL'] = 0.0
    return compute_final_premium(r)

sample = final_dataset.sample(min(30000, len(final_dataset)), random_state=42).copy()
sample['PREMIUM_NCD0'] = sample.apply(premium_at_zero_ncd, axis=1)
sample['NCD_DISCOUNT'] = 1 - sample['FINAL_PREMIUM_SST'] / sample['PREMIUM_NCD0']
max_ncd_disc = sample.loc[sample['NCD_LEVEL_PRICED'] == 0.55, 'NCD_DISCOUNT'].median()
validation_results.append({
    'Test': 'NCD Discount at max tier (controlled)',
    'Value': f"{max_ncd_disc*100:.1f}%",
    'Expected': '45-60%',
    'Pass': 0.45 <= max_ncd_disc <= 0.60
})

# 9. Comprehensive Premium Trend (2026 -> 2030)
comp_2026_avg = final_dataset[(final_dataset['SIM_YEAR'] == 2026) & (final_dataset['COVERAGE_TYPE'] == 'Comprehensive')]['FINAL_PREMIUM_SST'].mean()
comp_2030_avg = final_dataset[(final_dataset['SIM_YEAR'] == 2030) & (final_dataset['COVERAGE_TYPE'] == 'Comprehensive')]['FINAL_PREMIUM_SST'].mean()
comp_trend = comp_2030_avg / comp_2026_avg if comp_2026_avg > 0 else float('nan')
validation_results.append({
    'Test': 'Comp Premium Trend (2026 to 2030)',
    'Value': f"RM{comp_2026_avg:,.0f} to RM{comp_2030_avg:,.0f} ({comp_trend:.2f}x)",
    'Expected': 'Mild softening (0.80x-0.99x)',
    'Pass': 0.80 <= comp_trend < 1.00
})

validation_df = pd.DataFrame(validation_results)
print("\n" + validation_df.to_string(index=False))
all_pass = validation_df['Pass'].all()
print("\n" + "="*70)
if all_pass:
    print("ALL VALIDATION TESTS PASSED")
else:
    print("SOME VALIDATION TESTS FAILED - REVIEW ABOVE")
print("="*70)

# ---- Correlation Heatmap ----
numeric_cols = ['DRIVER_AGE', 'CAR_AGE', 'SUM_ASSURED', 'FINAL_PREMIUM_SST',
                'TOTAL_LOADING', 'NCD_LEVEL_PRICED', 'CLAIM_COUNT',
                'CLAIM_AMOUNT', 'CLAIM_LAMBDA', 'NCD_YEARS']
correlation_matrix = final_dataset[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1)
plt.title('Correlation Matrix - Key Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nCorrelation heatmap saved: images/correlation_heatmap.png")

# ============================================================================
# PART B: ENHANCED STATISTICAL TESTS
# ============================================================================
print("\n" + "="*70)
print("ENHANCED STATISTICAL VALIDATION")
print("="*70)

enhanced_results = []

# 1. Premium Log-Normal Fit (KS)
log_premiums = np.log(final_dataset['FINAL_PREMIUM_SST'])
ks_stat, ks_pval = kstest(log_premiums, norm(loc=log_premiums.mean(), scale=log_premiums.std()).cdf)
enhanced_results.append({
    'Test': 'Premium Log-Normal Fit (KS test)',
    'Statistic': f"KS={ks_stat:.4f}",
    'P-value': f"{ks_pval:.4e}",
    'Interpretation': 'Approx log-normal' if ks_stat < 0.1 else 'Mixture (TPO+Comp+NCD)',
    'Pass': ks_stat < 0.20
})
print("\n1. PREMIUM DISTRIBUTION FIT TEST (Log-Normal)")
print(f"   KS Statistic: {ks_stat:.4f}, p-value: {ks_pval:.4e}")
print(f"   (N={len(final_dataset):,}; threshold 0.15 given coverage/NCD mixture)")

# 2. Severity Gamma Fit by coverage
print("\n2. CLAIM SEVERITY DISTRIBUTION FIT (Gamma)")
print("-"*60)
for cov_type in ['Comprehensive', 'TPFT', 'TPO']:
    claims_subset = final_dataset[(final_dataset['COVERAGE_TYPE'] == cov_type) & (final_dataset['CLAIM_AMOUNT'] > 0)]['CLAIM_AMOUNT']
    if len(claims_subset) > 30:
        shape_fit, loc_fit, scale_fit = gamma_dist.fit(claims_subset, floc=0)
        ks_s, ks_p = kstest(claims_subset, gamma_dist(a=shape_fit, loc=loc_fit, scale=scale_fit).cdf)
        enhanced_results.append({
            'Test': f'Severity Gamma Fit ({cov_type})',
            'Statistic': f"shape={shape_fit:.2f}, KS={ks_s:.4f}",
            'P-value': f"{ks_p:.4e}",
            'Interpretation': f'Per-peril mixture, shape={shape_fit:.2f}',
            'Pass': ks_s < 0.20
        })
        print(f"   {cov_type}: shape={shape_fit:.2f}, scale={scale_fit:.0f}, KS={ks_s:.4f}")
    else:
        print(f"   {cov_type}: Insufficient claims (n={len(claims_subset)})")

# 3. NCD Distribution by Year
print("\n3. NCD DISTRIBUTION BY YEAR")
print("-"*60)
ncd_by_year = final_dataset.groupby('SIM_YEAR')['NCD_LEVEL_PRICED'].value_counts(normalize=True).unstack(fill_value=0)
ncd_by_year = ncd_by_year.reindex(columns=sorted(ncd_by_year.columns))
for year in sorted(final_dataset['SIM_YEAR'].unique()):
    yd = final_dataset[final_dataset['SIM_YEAR'] == year]
    print(f"   {year}: Avg NCD={yd['NCD_LEVEL_PRICED'].mean()*100:.1f}% | NCD=0%: {(yd['NCD_LEVEL_PRICED']==0.0).mean()*100:.1f}% | NCD=55%: {(yd['NCD_LEVEL_PRICED']==0.55).mean()*100:.1f}%")

ncd_2026_max = (final_dataset[final_dataset['SIM_YEAR']==2026]['NCD_LEVEL_PRICED']==0.55).mean()
ncd_2030_max = (final_dataset[final_dataset['SIM_YEAR']==2030]['NCD_LEVEL_PRICED']==0.55).mean()
enhanced_results.append({
    'Test': 'NCD: 55% tier grows over years',
    'Statistic': f"{ncd_2026_max*100:.1f}% to {ncd_2030_max*100:.1f}%",
    'P-value': '-',
    'Interpretation': 'Loyal claim-free policies accumulate',
    'Pass': ncd_2030_max > ncd_2026_max
})

# NCD distribution plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ncd_labels = sorted(final_dataset['NCD_LEVEL_PRICED'].unique())
ncd_year_data = []
for year in sorted(final_dataset['SIM_YEAR'].unique()):
    year_ncd = final_dataset[final_dataset['SIM_YEAR'] == year]['NCD_LEVEL_PRICED']
    row = {f'{n*100:.0f}%': (year_ncd == n).mean()*100 for n in ncd_labels}
    row['Year'] = year
    ncd_year_data.append(row)
ncd_plot_df = pd.DataFrame(ncd_year_data).set_index('Year')
ncd_plot_df.plot(kind='bar', stacked=True, ax=axes[0], colormap='YlOrRd_r')
axes[0].set_title('NCD Distribution by Year', fontsize=14, fontweight='bold')
axes[0].set_ylabel('% of Policies')
axes[0].legend(title='NCD %', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[0].set_xlabel('SIM_YEAR')

tier_labels = {0.0: '0%', 0.25: '25%', 0.30: '30%', 0.3833: '38%', 0.45: '45%', 0.55: '55%'}
final_dataset['NCD_TIER'] = final_dataset['NCD_LEVEL_PRICED'].map(tier_labels)
final_dataset.boxplot(column='FINAL_PREMIUM_SST', by='NCD_TIER', ax=axes[1])
axes[1].set_title('Premium Distribution by NCD Tier', fontsize=14, fontweight='bold')
axes[1].set_xlabel('NCD Tier')
axes[1].set_ylabel('Premium (RM)')
axes[1].get_figure().suptitle('')
plt.tight_layout()
plt.savefig('images/ncd_validation.png', dpi=150, bbox_inches='tight')
plt.close()

# 4. Loss Ratio by Segment
print("\n4. LOSS RATIO BY SEGMENT")
print("-"*60)
total_premium = final_dataset['FINAL_PREMIUM_SST'].sum()
total_incurred = final_dataset['CLAIM_AMOUNT'].sum()
overall_lr = total_incurred / total_premium
print("   By Coverage Type:")
for cov in ['Comprehensive', 'TPFT', 'TPO']:
    subset = final_dataset[final_dataset['COVERAGE_TYPE'] == cov]
    lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
    print(f"     {cov:14s}: LR={lr:.2%} (n={len(subset):,})")
final_dataset['age_band'] = pd.cut(final_dataset['DRIVER_AGE'],
                                   bins=[17, 25, 35, 50, 65, 100],
                                   labels=['18-25', '26-35', '36-50', '51-65', '66+'])
print("   By Age Band:")
for band in ['18-25', '26-35', '36-50', '51-65', '66+']:
    subset = final_dataset[final_dataset['age_band'] == band]
    if len(subset) > 0 and subset['FINAL_PREMIUM_SST'].sum() > 0:
        lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
        print(f"     {band:6s}: LR={lr:.2%} (n={len(subset):,})")
print("   By Region:")
for loc in ['Peninsular Malaysia', 'East Malaysia (Sabah, Sawarak & Labuan)']:
    subset = final_dataset[final_dataset['REGION'] == loc]
    lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
    print(f"     {loc[:12]:12s}: LR={lr:.2%}")
enhanced_results.append({
    'Test': 'Loss Ratio in Actuarial Range',
    'Statistic': f"LR={overall_lr:.2%}",
    'P-value': '-',
    'Interpretation': 'Typically 50-80% for motor',
    'Pass': 0.40 <= overall_lr <= 0.90
})

# 5. Severity Percentile Validation
print("\n5. SEVERITY PERCENTILE VALIDATION")
print("-"*60)
severity_targets = {
    'Comprehensive': {'mean': (4000, 10000), 'p95': (15000, 60000)},
    'TPO': {'mean': (8000, 30000), 'p95': (25000, 600000)},
    'TPFT': {'mean': (5000, 15000), 'p95': (20000, 120000)}
}
for cov_type, targets in severity_targets.items():
    claims_subset = final_dataset[(final_dataset['COVERAGE_TYPE'] == cov_type) & (final_dataset['CLAIM_AMOUNT'] > 0)]['CLAIM_AMOUNT']
    if len(claims_subset) > 10:
        mean_sev = claims_subset.mean()
        p95_sev = claims_subset.quantile(0.95)
        mean_pass = targets['mean'][0] <= mean_sev <= targets['mean'][1]
        p95_pass = targets['p95'][0] <= p95_sev <= targets['p95'][1]
        print(f"   {cov_type}: Mean=RM{mean_sev:,.0f} (target RM{targets['mean'][0]:,}-{targets['mean'][1]:,}) {'OK' if mean_pass else 'CHECK'}")
        print(f"     P95=RM{p95_sev:,.0f} (target RM{targets['p95'][0]:,}-{targets['p95'][1]:,}) {'OK' if p95_pass else 'CHECK'}")
        enhanced_results.append({'Test': f'Severity Mean ({cov_type})', 'Statistic': f"RM{mean_sev:,.0f}", 'P-value': '-', 'Interpretation': f'Target RM{targets["mean"][0]:,}-{targets["mean"][1]:,}', 'Pass': mean_pass})
        enhanced_results.append({'Test': f'Severity P95 ({cov_type})', 'Statistic': f"RM{p95_sev:,.0f}", 'P-value': '-', 'Interpretation': f'Target RM{targets["p95"][0]:,}-{targets["p95"][1]:,}', 'Pass': p95_pass})

# 6. Longitudinal Consistency (POLID tracking)
print("\n6. LONGITUDINAL CONSISTENCY")
print("-"*60)
ph_counts = final_dataset.groupby('POLID')['SIM_YEAR'].nunique()
print(f"   Unique policies: {len(ph_counts):,}")
print(f"   Policies with 1 year: {(ph_counts == 1).sum():,}")
print(f"   Policies with 2+ years: {(ph_counts >= 2).sum():,}")
print(f"   Policies with all 5 years: {(ph_counts == 5).sum():,}")
multi_year_phs = ph_counts[ph_counts >= 2].index[:1000]
age_errors = 0
car_errors = 0
ncd_reset_failures = 0
total_claims_checked = 0
for ph_id in multi_year_phs:
    ph = final_dataset[final_dataset['POLID'] == ph_id].sort_values('SIM_YEAR')
    for i in range(1, len(ph)):
        year_diff = ph.iloc[i]['SIM_YEAR'] - ph.iloc[i-1]['SIM_YEAR']
        age_diff = ph.iloc[i]['DRIVER_AGE'] - ph.iloc[i-1]['DRIVER_AGE']
        car_diff = ph.iloc[i]['CAR_AGE'] - ph.iloc[i-1]['CAR_AGE']
        if age_diff != year_diff:
            age_errors += 1
        if car_diff != year_diff and not (car_diff == 0 and ph.iloc[i]['CAR_AGE'] == 10):
            car_errors += 1
        prev_claim = ph.iloc[i-1]['CLAIM_OCCURRED']
        curr_ncd = ph.iloc[i]['NCD_LEVEL_PRICED']
        if prev_claim:
            total_claims_checked += 1
            if curr_ncd > 0:
                ncd_reset_failures += 1
age_ok = age_errors == 0
car_ok = car_errors == 0
ncd_ok = ncd_reset_failures == 0
print(f"\n   Age consistency (sample {len(multi_year_phs):,}): errors={age_errors} {'OK' if age_ok else 'CHECK'}")
print(f"   Car-age consistency (cap at 10 allowed): errors={car_errors} {'OK' if car_ok else 'CHECK'}")
print(f"   NCD reset on claim: failures={ncd_reset_failures}/{total_claims_checked} {'OK' if ncd_ok else 'CHECK'}")
enhanced_results.append({'Test': 'Longitudinal: Age Consistency', 'Statistic': f"{age_errors} errors", 'P-value': '-', 'Interpretation': 'DRIVER_AGE +1 per year', 'Pass': age_ok})
enhanced_results.append({'Test': 'Longitudinal: Car-Age Consistency', 'Statistic': f"{car_errors} errors", 'P-value': '-', 'Interpretation': 'CAR_AGE +1/yr (cap 10)', 'Pass': car_ok})
enhanced_results.append({'Test': 'NCD Reset on Claim', 'Statistic': f"{ncd_reset_failures}/{total_claims_checked} failures", 'P-value': '-', 'Interpretation': 'Claim in year N -> NCD 0 next year', 'Pass': ncd_ok})

# 7b. TPFT claim frequency between TPO and Comprehensive
comp_freq_c = final_dataset[final_dataset['COVERAGE_TYPE'] == 'Comprehensive']['CLAIM_OCCURRED'].mean()
tpo_freq_c = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPO']['CLAIM_OCCURRED'].mean()
tpft_freq_c = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPFT']['CLAIM_OCCURRED'].mean()
enhanced_results.append({'Test': 'TPFT Claim Frequency between TPO & Comp', 'Statistic': f"{tpo_freq_c:.1%} < {tpft_freq_c:.1%} < {comp_freq_c:.1%}", 'P-value': '-', 'Interpretation': 'TPFT covers TP + fire/theft only', 'Pass': tpo_freq_c < tpft_freq_c < comp_freq_c})

# 7. Young Adults vs older drivers
print("\n7. GEN Z vs NON-GEN Z COMPARISON")
print("-"*60)
gen_z = final_dataset[final_dataset['DRIVER_AGE'] <= 27]
non_gen_z = final_dataset[final_dataset['DRIVER_AGE'] > 27]
gen_z_pct = len(gen_z) / len(final_dataset) * 100
gz_claim = (gen_z['CLAIM_COUNT'] > 0).mean()
nz_claim = (non_gen_z['CLAIM_COUNT'] > 0).mean()
gz_prem = gen_z['FINAL_PREMIUM_SST'].mean()
nz_prem = non_gen_z['FINAL_PREMIUM_SST'].mean()
print(f"   Young Adults share: {gen_z_pct:.1f}%")
print(f"   Claim rate: Young Adults={gz_claim*100:.1f}% vs Older={nz_claim*100:.1f}%")
print(f"   Avg premium: Young Adults=RM{gz_prem:,.0f} vs Older=RM{nz_prem:,.0f}")
enhanced_results.append({'Test': 'Young Adults Share', 'Statistic': f"{gen_z_pct:.1f}%", 'P-value': '-', 'Interpretation': 'Target 25-40%', 'Pass': 25 <= gen_z_pct <= 40})
enhanced_results.append({'Test': 'Young Adults Higher Claim Rate', 'Statistic': f"{gz_claim*100:.1f}% vs {nz_claim*100:.1f}%", 'P-value': '-', 'Interpretation': 'Young drivers claim more', 'Pass': gz_claim > nz_claim})

# 8. Claim Count Poisson Fit (Var/Mean)
print("\n8. CLAIM COUNT DISTRIBUTION")
print("-"*60)
claim_counts = final_dataset['CLAIM_COUNT']
mean_claims = claim_counts.mean()
var_mean = claim_counts.var() / mean_claims
print(f"   Mean: {mean_claims:.4f}, Var/Mean: {var_mean:.3f} (1.0 = perfect Poisson)")
enhanced_results.append({'Test': 'Claim Count Poisson Fit (Var/Mean)', 'Statistic': f"{var_mean:.3f}", 'P-value': '-', 'Interpretation': 'Close to 1.0 (heterogeneity inflates)', 'Pass': 0.7 <= var_mean <= 1.6})

# 9. Log-Premium Shape
print("\n9. PREMIUM DISTRIBUTION SHAPE")
print("-"*60)
skewness = log_premiums.skew()
kurtosis = log_premiums.kurtosis()
dagostino_stat, dagostino_p = normaltest(log_premiums.sample(min(5000, len(log_premiums)), random_state=42))
print(f"   Log-premium skewness: {skewness:.3f} (target |skew| < 1.5)")
print(f"   Log-premium kurtosis: {kurtosis:.3f}")
print(f"   D'Agostino-Pearson: stat={dagostino_stat:.2f}, p={dagostino_p:.4e}")
enhanced_results.append({'Test': 'Log-Premium Skewness', 'Statistic': f"{skewness:.3f}", 'P-value': '-', 'Interpretation': '|skew| < 1.5 (tariff-fixed TPO flat premium widens left mass)', 'Pass': abs(skewness) < 1.5})

# Summary
print("\n" + "="*70)
print("ENHANCED VALIDATION SUMMARY")
print("="*70)
enhanced_df = pd.DataFrame(enhanced_results)
print("\n" + enhanced_df.to_string(index=False))
pass_count = enhanced_df['Pass'].sum()
total_count = len(enhanced_df)
print(f"\nResult: {pass_count}/{total_count} tests passed")
if pass_count == total_count:
    print("ALL ENHANCED VALIDATION TESTS PASSED")
else:
    print(f"{total_count - pass_count} test(s) failed - review above")
print("="*70)


In [ ]:
# Cohort Analysis Summary

def analyze_cohort(df_cohort):
    """Generate summary statistics from cohort simulation."""
    summary = []
    
    for year in sorted(df_cohort['SIM_YEAR'].unique()):
        year_df = df_cohort[df_cohort['SIM_YEAR'] == year]
        summary.append({
            'Year': year,
            'Active_Policies': len(year_df),
            'Total_Claims': year_df['CLAIM_COUNT'].sum(),
            'Avg_Claims_Per_Policy': year_df['CLAIM_COUNT'].mean(),
            'Total_Claim_Amount': year_df['CLAIM_AMOUNT'].sum(),
            'Avg_Claim_Amount': year_df.loc[
                year_df['CLAIM_COUNT'] > 0, 'CLAIM_AMOUNT'
            ].mean() if year_df['CLAIM_COUNT'].sum() > 0 else 0,
            'Avg_NCD_Level': year_df['NCD_LEVEL'].mean(),
            'Retention_Rate': year_df['RENEWED'].mean()
            if 'RENEWED' in year_df.columns else float('nan'),
            'Avg_Final_Premium': year_df['FINAL_PREMIUM_SST'].mean()
        })
    
    return pd.DataFrame(summary)

# Generate and display cohort summary
cohort_summary = analyze_cohort(cohort_results)
print('=== n-Year Cohort Evolution Summary ===\n')
print(cohort_summary.to_string(index=False))
print(f"\n=== Key Findings ===")
print(f"Total claims over 5 years: {cohort_results['CLAIM_COUNT'].sum():.0f}")
print(f"Total claim cost: RM{cohort_results['CLAIM_AMOUNT'].sum():,.2f}")
print(f"Final avg NCD level: {cohort_results[cohort_results['SIM_YEAR']==2028]['NCD_LEVEL'].mean():.2%}")
print(f"Loss ratio: {cohort_results['CLAIM_AMOUNT'].sum() / cohort_results['FINAL_PREMIUM_SST'].sum():.2%}")

In [ ]:

# ============================================================
# Individual Policy Trajectories (premium evolution over years)
# ============================================================

def plot_policy_trajectories(df_cohort, n_policies=20, min_years=3,
                             out='images/policy-trajectories.png', seed=42):
    """Plot premium + priced-NCD trajectories for random multi-year policies."""
    years_per_polid = df_cohort.groupby('POLID')['SIM_YEAR'].nunique()
    eligible = years_per_polid[years_per_polid >= min_years].index.tolist()
    rng = np.random.RandomState(seed)
    picks = [str(p) for p in rng.choice(
        eligible, size=min(n_policies, len(eligible)), replace=False
    )]

    sel = df_cohort[df_cohort['POLID'].isin(picks)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for pid in picks:
        p = sel[sel['POLID'] == pid].sort_values('SIM_YEAR')
        axes[0].plot(p['SIM_YEAR'], p['FINAL_PREMIUM_SST'], marker='o',
                     label=f"{pid[:12]}...")
        axes[1].plot(p['SIM_YEAR'], p['NCD_LEVEL_PRICED'], marker='s')
    axes[0].set_title('Final premium by year')
    axes[0].set_xlabel('SIM_YEAR')
    axes[0].set_ylabel('FINAL_PREMIUM_SST (RM)')
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)
    axes[1].set_title('NCD used for pricing by year')
    axes[1].set_xlabel('SIM_YEAR')
    axes[1].set_ylabel('NCD_LEVEL_PRICED')
    axes[1].grid(alpha=0.3)
    fig.tight_layout()

    os.makedirs(os.path.dirname(out), exist_ok=True)
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Trajectory plot saved: {out}")

    print('\n=== Selected policy trajectories (year-by-year) ===')
    cols = ['POLID', 'SIM_YEAR', 'COVERAGE_TYPE', 'DRIVER_AGE_CAT', 'DRIVER_AGE',
            'CAR_AGE', 'NCD_LEVEL_PRICED', 'FINAL_PREMIUM_SST', 'CLAIM_OCCURRED']
    print(sel.sort_values(['POLID', 'SIM_YEAR'])[cols].to_string(index=False))


plot_policy_trajectories(cohort_results)


In [ ]:
# ============================================================================
# OVERTHINKER-STYLE EDA (adapted from reference Section 7)
# ============================================================================

print("="*70)
print("GENERATING ACTUARIAL EDA REPORTS")
print("="*70)

# 1. Actuarial Premium Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
pivot_agencd = final_dataset.groupby(['age_band', 'NCD_TIER'], observed=True)['FINAL_PREMIUM_SST'].median().unstack()
sns.heatmap(pivot_agencd, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0],
            cbar_kws={'label': 'Median Premium (RM)'})
axes[0].set_title('Median Premium: Age Band vs NCD Tier')
axes[0].set_xlabel('NCD Tier')
axes[0].set_ylabel('Driver Age Band')

pivot_covloc = final_dataset.groupby(['COVERAGE_TYPE', 'REGION'], observed=True)['FINAL_PREMIUM_SST'].median().unstack()
pivot_covloc.columns = [c.replace(' Malaysia (Sabah, Sawarak & Labuan)', '').replace(' Malaysia', '') for c in pivot_covloc.columns]
pivot_covloc = pivot_covloc.reindex(['Comprehensive', 'TPFT', 'TPO'])
sns.heatmap(pivot_covloc, annot=True, fmt='.0f', cmap='Blues', ax=axes[1],
            cbar_kws={'label': 'Median Premium (RM)'})
axes[1].set_title('Median Premium: Coverage vs Region')
plt.tight_layout()
plt.savefig('images/eda_premium_heatmaps.png', dpi=150, bbox_inches='tight')
plt.close()

# 2. Risk Profile: Frequency & Loss Ratio by Age Band
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
freq_by_age = final_dataset.groupby('age_band', observed=True)['CLAIM_COUNT'].apply(lambda x: (x > 0).mean() * 100)
sns.barplot(x=freq_by_age.index, y=freq_by_age.values, ax=axes[0], palette='viridis')
axes[0].axhline((final_dataset['CLAIM_COUNT'] > 0).mean() * 100, ls='--', color='red', label='Portfolio Average')
axes[0].set_title('Claim Frequency by Age Band (U-Shaped Risk)')
axes[0].set_ylabel('Claim Frequency (%)')
axes[0].legend()

lr_by_age = final_dataset.groupby('age_band', observed=True).apply(
    lambda x: (x['CLAIM_AMOUNT'].sum() / x['FINAL_PREMIUM_SST'].sum()) * 100
)
sns.barplot(x=lr_by_age.index, y=lr_by_age.values, ax=axes[1], palette='magma')
axes[1].axhline((final_dataset['CLAIM_AMOUNT'].sum() / final_dataset['FINAL_PREMIUM_SST'].sum()) * 100,
                ls='--', color='red', label='Portfolio Average')
axes[1].set_title('Loss Ratio by Age Band')
axes[1].set_ylabel('Loss Ratio (%)')
axes[1].legend()
plt.tight_layout()
plt.savefig('images/eda_risk_profile.png', dpi=150, bbox_inches='tight')
plt.close()

# 3. Severity Tails (Log-Scale)
plt.figure(figsize=(10, 6))
claimants = final_dataset[final_dataset['CLAIM_AMOUNT'] > 0]
sns.histplot(data=claimants, x=np.log1p(claimants['CLAIM_AMOUNT']),
             hue='COVERAGE_TYPE', hue_order=['Comprehensive', 'TPFT', 'TPO'],
             bins=50, kde=True, palette='Set2', alpha=0.6, element='step')
plt.title('Log-Scale Claim Severity Distribution (Fat Tails)')
plt.xlabel('log(1 + Claim Amount)')
plt.ylabel('Count of Claims')
plt.savefig('images/eda_severity_tails.png', dpi=150, bbox_inches='tight')
plt.close()

# 4. Severity Distributions by Coverage (Gamma overlay)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, cov in enumerate(['Comprehensive', 'TPFT', 'TPO']):
    claims_subset = final_dataset[(final_dataset['COVERAGE_TYPE'] == cov) & (final_dataset['CLAIM_AMOUNT'] > 0)]['CLAIM_AMOUNT']
    if len(claims_subset) > 10:
        claims_subset.hist(bins=50, ax=axes[i], color=['skyblue', 'coral', 'seagreen'][i],
                           edgecolor='black', alpha=0.7, density=True)
        shape_f, loc_f, scale_f = gamma_dist.fit(claims_subset, floc=0)
        x = np.linspace(0, claims_subset.quantile(0.99), 200)
        axes[i].plot(x, gamma_dist.pdf(x, shape_f, loc_f, scale_f), 'r-', lw=2,
                     label=f'Gamma fit (k={shape_f:.1f})')
        axes[i].set_title(f'{cov} Severity Distribution', fontsize=13, fontweight='bold')
        axes[i].set_xlabel('Claim Amount (RM)')
        axes[i].legend()
        axes[i].axvline(claims_subset.mean(), color='black', linestyle='--', alpha=0.5, label='Mean')
        axes[i].axvline(claims_subset.quantile(0.95), color='red', linestyle=':', alpha=0.5, label='P95')
plt.tight_layout()
plt.savefig('images/severity_distributions.png', dpi=150, bbox_inches='tight')
plt.close()

# 5. Longitudinal Trends
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
year_stats = final_dataset.groupby('SIM_YEAR').agg({
    'FINAL_PREMIUM_SST': 'mean',
    'NCD_LEVEL_PRICED': 'mean',
    'CLAIM_COUNT': lambda x: (x > 0).mean()
}).reset_index()
axes[0].plot(year_stats['SIM_YEAR'], year_stats['FINAL_PREMIUM_SST'], 'b-o', linewidth=2, markersize=8)
axes[0].set_title('Average Premium by Year', fontsize=14, fontweight='bold')
axes[0].set_xlabel('SIM_YEAR')
axes[0].set_ylabel('Avg Premium (RM)')
axes[0].grid(True, alpha=0.3)
ax2 = axes[0].twinx()
ax2.plot(year_stats['SIM_YEAR'], year_stats['NCD_LEVEL_PRICED']*100, 'r--s', linewidth=2, markersize=6, alpha=0.7)
ax2.set_ylabel('Avg NCD %', color='red')

for label, age_min, age_max, color in [('Gen Z (18-27)', 18, 27, 'red'),
                                       ('Prime (28-50)', 28, 50, 'blue'),
                                       ('Senior (51+)', 51, 100, 'green')]:
    subset = final_dataset[(final_dataset['DRIVER_AGE'] >= age_min) & (final_dataset['DRIVER_AGE'] <= age_max)]
    rates = subset.groupby('SIM_YEAR')['CLAIM_COUNT'].apply(lambda x: (x > 0).mean() * 100)
    axes[1].plot(rates.index, rates.values, '-o', label=label, color=color, linewidth=2)
axes[1].set_title('Claim Rate by Year & Age Group', fontsize=14, fontweight='bold')
axes[1].set_xlabel('SIM_YEAR')
axes[1].set_ylabel('Claim Rate (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('images/longitudinal_trends.png', dpi=150, bbox_inches='tight')
plt.close()

print("EDA visualisations generated and saved:")
print("  images/correlation_heatmap.png, images/ncd_validation.png")
print("  images/eda_premium_heatmaps.png, images/eda_risk_profile.png")
print("  images/eda_severity_tails.png, images/severity_distributions.png")
print("  images/longitudinal_trends.png")


In [ ]:
# ============================================================================
# EV TREND MARKET ANALYSIS - descriptive EV/ICE split + adoption scenarios
# Theory (fair-value): premium tracks SA, own-risk severity tracks SA, TPBI/TPPD
# are SA-independent -> EV LR ~ ICE LR (slightly lower) as EV share rises.
# ============================================================================

# ---- 1. Descriptive: EV vs ICE by year (base run) ----
_res = cohort_results
_is_ent = _res['POLID'].str.startswith('ENT')

ev_all = _res.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())
ev_ent = _res[_is_ent].groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())


def _ev_ice_metrics(g):
    out = {}
    for name, sub in [('EV', g[g['VEHICLE_TYPE'] == 'EV']),
                      ('ICE', g[g['VEHICLE_TYPE'] == 'ICE'])]:
        prem = sub['FINAL_PREMIUM_SST'].sum()
        ncl = sub['CLAIM_COUNT'].sum()
        out[name + '_lr'] = sub['CLAIM_AMOUNT'].sum() / prem if prem > 0 else float('nan')
        out[name + '_freq'] = sub['CLAIM_OCCURRED'].mean()
        out[name + '_sev'] = (sub['CLAIM_AMOUNT'].sum() / ncl) if ncl > 0 else float('nan')
        out[name + '_prem'] = sub['FINAL_PREMIUM_SST'].mean()
        out[name + '_ret'] = sub['RENEWED'].mean()
        out[name + '_ncd'] = sub['NCD_LEVEL'].mean()
    return pd.Series(out)


ev_ice = _res.groupby('SIM_YEAR').apply(_ev_ice_metrics)

print('=== EV adoption (base run) ===')
print('  EV share overall : {:.1%} -> {:.1%}'.format(ev_all.iloc[0], ev_all.iloc[-1]))
print('  EV share entrants: {:.1%} -> {:.1%}'.format(ev_ent.iloc[0], ev_ent.iloc[-1]))
print('\n=== EV vs ICE by year (base run) ===')
print(ev_ice.round(4).to_string())
print('\n  Final-year EV LR vs ICE LR: {:.1%} vs {:.1%}'.format(
    ev_ice['EV_lr'].iloc[-1], ev_ice['ICE_lr'].iloc[-1]))

# ---- 2. Adoption scenarios (same book + seed, only EV ramp varies) ----
_SCENARIOS = {
    'Conservative': {2026: 0.03, 2030: 0.06, 2035: 0.12, 2040: 0.18, 2045: 0.25},
    'Baseline':     dict(COHORT_CONFIG['ev_share_by_year']),
    'Aggressive':   {2026: 0.08, 2030: 0.20, 2035: 0.40, 2040: 0.65, 2045: 0.85},
}

_scen_rows = []
_scen_curves = {}
for _name, _sched in _SCENARIOS.items():
    _cfg = deep_update(COHORT_CONFIG, {'ev_share_by_year': _sched})
    _sim = simulate_cohort(df, n_years=20, new_entrants_per_year=None,
                           seed=42, cfg=_cfg, verbose=False)
    _sim = price_book(_sim, 'telem', _cfg)
    _final = _sim[_sim['SIM_YEAR'] == _sim['SIM_YEAR'].max()]
    _scen_rows.append({
        'scenario': _name,
        'final_ev_share': (_final['VEHICLE_TYPE'] == 'EV').mean(),
        'overall_lr': _sim['CLAIM_AMOUNT'].sum() / _sim['FINAL_PREMIUM_SST'].sum(),
        'n_policy_years': len(_sim),
    })
    _scen_curves[_name] = _sim.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(
        lambda s: (s == 'EV').mean())
scen_df = pd.DataFrame(_scen_rows)

print('\n=== EV adoption scenarios (same book, seed 42) ===')
print(scen_df.round(4).to_string(index=False))
lr_ord = scen_df.sort_values('final_ev_share')['overall_lr']
print('  LR ordering (fair-value: more EV -> slightly lower LR):',
      'OK' if list(lr_ord) == sorted(lr_ord, reverse=True) else 'CHECK')

# ---- 3. Charts ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(ev_all.index, ev_all.values, marker='o', label='Overall')
ax.plot(ev_ent.index, ev_ent.values, marker='s', label='Entrants')
ax.set_title('EV share over time (base run)')
ax.set_xlabel('Year')
ax.set_ylabel('EV share')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(ev_ice.index, ev_ice['EV_lr'].values, marker='o', label='EV')
ax.plot(ev_ice.index, ev_ice['ICE_lr'].values, marker='s', label='ICE')
ax.set_title('Loss ratio by vehicle type (base run)')
ax.set_xlabel('Year')
ax.set_ylabel('Loss ratio')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[2]
for _name, _curve in _scen_curves.items():
    _lr = scen_df.loc[scen_df['scenario'] == _name, 'overall_lr'].iloc[0]
    ax.plot(_curve.index, _curve.values, marker='o',
            label=_name + ' (LR {:.1%})'.format(_lr))
ax.set_title('EV share by scenario')
ax.set_xlabel('Year')
ax.set_ylabel('EV share')
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.savefig('images/ev_analysis.png', dpi=150)
print('\nChart saved: data/ev_analysis.png')


# Pricing Model Progression: Tariff -> GLM -> GLM+Telematics

Three **identical initial cohorts** (same seed 42 -> same policies, same latent `BEHAVIOR_RISK`
driving claims, same simulated claims) are priced under three regimes. Only **pricing** differs:

- **Act 1 - Traditional Tariff**: graduated Motor Tariff premium, no behavioral risk priced.
  Behaviour (mean 1.10 on claim frequency) is unpriced -> rate-*inadequate* book (portfolio LR up).
- **Act 2 - GLM** (traditional rating factors): Poisson GLM calibrated on the tariff book's own
  experience -> rate-*adequate* (portfolio LR down), but it cannot see behaviour -> risk is
  mis-*equitably* spread.
- **Act 3 - GLM + Telematics** (teammate's `BPM_Telemetrics_approach` features): additionally sees
  `telematics_score` (a monotone proxy of behaviour) -> prices behaviour too -> best risk equity
  (highest premium-count correlation) and lowest retained-book LR after declining the worst 15% of
  policies.

Models are fitted on **entrant-only policy-years** (`COHORT_YEAR == SIM_YEAR`, entry-NCD semantics -
no claim-history endogeneity) and a 60% random policy split; the fitted models then price the full
book. The NCD feature used is `NCD_LEVEL_PRICED` (the level actually used at pricing time, one-year
lag) - not the end-of-year post-update level, which would leak claim history. Telemetry features:
hard braking / speeding / night driving -> composite `telematics_score`
-> `BEHAVIOR_RISK` (rank-mapped, mean 1.10) multiplies claim frequency. Premium formulas:
tariff = unchanged tariff; glm = `pred_freq x coverage-avg-severity x expense_loading x (1.1^flood) x (1.1^theft)`;
telem = same with `x telemetric_load`. Severity is coverage-specific, defined as expected claim
cost per claim `E[CLAIM_AMOUNT]/E[CLAIM_COUNT]` (not mean severity given a claim, which overstates
per-claim cost): TPFT RM11.5k / TPO RM9.5k / Comprehensive RM6.2k so each segment is priced at its
own cost level.

In [ ]:
# ============================================================================
# PRICING PROGRESSION RUN - three identical cohorts, three regimes, same seed
# ============================================================================

from sklearn.linear_model import PoissonRegressor
from scipy.stats import spearmanr

GLM_FEATURES = ['DRIVER_AGE', 'CAR_AGE', 'NCD_LEVEL_PRICED', 'VEHICLE_TYPE',
                'COVERAGE_TYPE', 'FLOOD_RISK', 'THEFT_RISK', 'REGION']
TELEM_FEATURES = GLM_FEATURES + ['telematics_score']


def _book_lr(res):
    return res['CLAIM_AMOUNT'].sum() / res['FINAL_PREMIUM_SST'].sum() * 100


def _book_spearman(res):
    p = res.groupby('POLID').agg(c=('CLAIM_COUNT', 'sum'),
                                 prem=('FINAL_PREMIUM_SST', 'sum'))
    return spearmanr(p['prem'], p['c']).statistic


def _retained_lr(res, decline_pct):
    p = res.groupby('POLID').agg(claims=('CLAIM_AMOUNT', 'sum'),
                                 prem=('FINAL_PREMIUM_SST', 'sum')).reset_index()
    k = int(np.floor(len(p) * decline_pct))
    p = p.sort_values('prem', ascending=False).iloc[k:]
    return p['claims'].sum() / p['prem'].sum() * 100


# Re-price the single baseline book (Ch.6 Baseline Run) under three regimes.
# No re-simulation: only pricing differs (same policies, same claims).
cohort_results_tariff = price_book(cohort_results, 'tariff', COHORT_CONFIG)
cohort_results_glm    = price_book(cohort_results, 'glm', COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED)
cohort_results_telem  = price_book(cohort_results, 'telem', COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED)
# NOTE: global cohort_results stays the TARIFF reference book (set in Baseline Run);
# it is NOT overwritten here, so downstream single-method sections stay consistent.

avg_sev = (cohort_results['CLAIM_AMOUNT'].sum()
           / cohort_results['CLAIM_COUNT'].sum())
avg_sev_by_cov = cohort_results.groupby('COVERAGE_TYPE').apply(
    lambda d: d['CLAIM_AMOUNT'].sum() / d['CLAIM_COUNT'].sum(), include_groups=False).to_dict()
exp_loading = COHORT_CONFIG.get('expense_loading', 1.40)
telem_load = COHORT_CONFIG.get('telemetric_load', 1.0)
print(f'avg severity/claim: RM{avg_sev:,.0f} | by coverage: ' +
      ', '.join(f'{k} RM{v:,.0f}' for k, v in sorted(avg_sev_by_cov.items())) +
      f' | expense_loading: {exp_loading:.2f} | telemetric_load: {telem_load:.2f}')

# --- summary table: full-book LR + premium-count Spearman rho + retained LR
#     (decline worst 15% of policies) ---
rows = []
for name, res in [('Tariff', cohort_results_tariff),
                  ('GLM', cohort_results_glm),
                  ('GLM+Telematics', cohort_results_telem)]:
    rows.append({'Regime': name,
                 'Portfolio LR (%)': round(_book_lr(res), 2),
                 'Prem-Count rho': round(_book_spearman(res), 4),
                 'Retained LR 15% (%)': round(_retained_lr(res, 0.15), 2)})
lr_df = pd.DataFrame(rows)
print('=== PRICING PROGRESSION (same book, same seed; only pricing differs) ===')
print(lr_df.to_string(index=False))
print()
print('=== compare_pricing() [engine API] ===')
print(compare_pricing(cohort_results, cfg=COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED).to_string(index=False))

# --- Test 13 checks ---
_passed = []
def _check(name, ok, info=''):
    _passed.append(ok)
    print(f'[{"PASS" if ok else "FAIL"}] {name} {info}')

r = cohort_results_tariff
cb, cn = r.loc[r['CLAIM_OCCURRED'], 'BEHAVIOR_RISK'].mean(), r.loc[~r['CLAIM_OCCURRED'], 'BEHAVIOR_RISK'].mean()
_check('13a. BEHAVIOR_RISK materially higher for claimants',
       cb - cn >= 0.005,
       f"({cb:.3f} vs {cn:.3f}, diff={cb - cn:+.3f})")
sb, sn = r.loc[r['CLAIM_OCCURRED'], 'telematics_score'].mean(), r.loc[~r['CLAIM_OCCURRED'], 'telematics_score'].mean()
_check('13b. telematics_score lower for claimants',
       sb < sn, f"({sb:.1f} vs {sn:.1f})")
_check('13c. portfolio LR: telem < tariff',
       lr_df.loc[2, 'Portfolio LR (%)'] < lr_df.loc[0, 'Portfolio LR (%)'],
       f"({lr_df.loc[2, 'Portfolio LR (%)']:.2f}% < {lr_df.loc[0, 'Portfolio LR (%)']:.2f}%)")
_check('13d. retained LR: GLM & telem << tariff; telem within 0.5pp of GLM',
       lr_df.loc[1, 'Retained LR 15% (%)'] < lr_df.loc[0, 'Retained LR 15% (%)']
       and lr_df.loc[2, 'Retained LR 15% (%)'] <= lr_df.loc[1, 'Retained LR 15% (%)'] + 0.5,
       f"(tariff {lr_df.loc[0, 'Retained LR 15% (%)']:.2f}% | glm {lr_df.loc[1, 'Retained LR 15% (%)']:.2f}% | telem {lr_df.loc[2, 'Retained LR 15% (%)']:.2f}%)")
_check('13e. premium-count correlation: telem > glm (prices behavior)',
       lr_df.loc[2, 'Prem-Count rho'] > lr_df.loc[1, 'Prem-Count rho'],
       f"({lr_df.loc[2, 'Prem-Count rho']:.4f} > {lr_df.loc[1, 'Prem-Count rho']:.4f})")

def _tier_lr(res):
    d = res.copy()
    d['telem_bin'] = pd.cut(d['telematics_score'],
                            bins=[0, 50, 70, 85, 100],
                            labels=['<50 (High Risk)', '50-70 (Moderate)',
                                    '70-85 (Low Risk)', '85-100 (Safe)'])
    return d.groupby('telem_bin', observed=False).apply(
        lambda g: g['CLAIM_AMOUNT'].sum() / g['FINAL_PREMIUM_SST'].sum() * 100)

tier_glm = _tier_lr(cohort_results_glm)
tier_telem = _tier_lr(cohort_results_telem)
# 13f (reframed): GLM is blind to telematics, so its premium does NOT track
# telematics_score; telem's premium is risk-based (negative score-premium corr).
def _score_corr(res):
    return spearmanr(res['FINAL_PREMIUM_SST'], res['telematics_score']).correlation
_c_glm = _score_corr(cohort_results_glm)
_c_telem = _score_corr(cohort_results_telem)
_check('13f. telem premium tracks telematics_score (risk-based); GLM ~blind',
       _c_telem < _c_glm - 0.1,
       f"(glm {_c_glm:+.3f} | telem {_c_telem:+.3f})")
print(f'Pricing-progression checks: {sum(_passed)}/{len(_passed)} passed')

# --- charts ---
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
colors = ['#94a3b8', '#f59e0b', '#2563eb']
axes[0].bar(lr_df['Regime'], lr_df['Portfolio LR (%)'], color=colors)
axes[0].set_ylabel('Portfolio Loss Ratio (%)')
axes[0].set_title('Full-Book LR by Pricing Regime')
for i, v in enumerate(lr_df['Portfolio LR (%)']):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')
axes[1].bar(lr_df['Regime'], lr_df['Retained LR 15% (%)'], color=colors)
axes[1].set_ylabel('Retained Book LR (%)')
axes[1].set_title('Retained-Book LR after Declining Worst 15% of Policies')
for i, v in enumerate(lr_df['Retained LR 15% (%)']):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')
axes[2].bar(lr_df['Regime'], lr_df['Prem-Count rho'], color=colors)
axes[2].set_ylabel('Spearman rho (premium vs claim count)')
axes[2].set_title('Pricing Accuracy on Claim Frequency (higher = better)')
for i, v in enumerate(lr_df['Prem-Count rho']):
    axes[2].text(i, v + 0.004, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('images/pricing_progression.png', dpi=150, bbox_inches='tight')
plt.show()

# --- discrimination check: LR by telematics risk tier (GLM blind vs telem priced) ---
print('=== LR by telematics risk tier (portfolio LR ~65-75%) ===')
print('GLM (blind to behavior):')
print(tier_glm.round(1).to_string())
print('GLM+Telematics (prices behavior):')
print(tier_telem.round(1).to_string())



## 8 - Monte Carlo Stress Test

Scenario x seed x pricing-model grid (`run_monte_carlo`) for uncertainty around the baseline.

In [ ]:
# ============================================================================
# MONTE CARLO API - scenario x seed grid, summary metrics + percentiles
# Vectorized engine + per-seed local RNG -> safe for parallel seeds.
# ============================================================================

def deep_update(base, overrides):
    """Deep-copy base and recursively merge overrides (None-safe)."""
    out = copy.deepcopy(base)
    if not overrides:
        return out
    for k, v in overrides.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_update(out[k], v)
        else:
            out[k] = v
    return out


def summarize(results_df):
    """One trajectory -> scalar metrics + per-year loss-ratio series."""
    premium = results_df['FINAL_PREMIUM_SST'].sum()
    claims = results_df['CLAIM_AMOUNT'].sum()
    metrics = {
        'overall_lr': claims / premium if premium > 0 else np.nan,
        'claim_freq': results_df['CLAIM_OCCURRED'].mean(),
        'avg_premium': results_df['FINAL_PREMIUM_SST'].mean(),
        'retention': results_df['RENEWED'].mean(),
        'n_policy_years': len(results_df),
    }

    def _yr_lr(g):
        p = g['FINAL_PREMIUM_SST'].sum()
        return g['CLAIM_AMOUNT'].sum() / p if p > 0 else np.nan

    yearly = results_df.groupby('SIM_YEAR').apply(_yr_lr)
    return metrics, yearly


def _run_one(cfg, seed, n_years, new_entrants_per_year, pricing_model='tariff',
             book_seed=COHORT_CONFIG['seed']):
    df0 = generate_dataset(cfg, seed=book_seed)
    res = simulate_cohort(df0, n_years=n_years,
                          new_entrants_per_year=new_entrants_per_year,
                          seed=seed, cfg=cfg, verbose=False)
    res = price_book(res, pricing_model, cfg)
    return summarize(res)


def run_monte_carlo(overrides=None, seeds=(42,), n_years=20,
                    new_entrants_per_year=None, n_jobs=4, book_seed=None,
                    pricing_models=('tariff',)):
    """Run a (scenario x seed) Monte Carlo grid.

    Args:
        overrides: None | dict | list[dict] - assumption patches on COHORT_CONFIG
        seeds: int | list[int] - independent replications per scenario
        n_years: simulation horizon
        new_entrants_per_year: static entrants/year override (None -> dynamic
                                growth-based count from cfg)
        n_jobs: parallel workers (threading backend; numpy releases the GIL)
        pricing_models: pricing models evaluated per (scenario, seed);
                        e.g. ('tariff', 'glm', 'telem')
        book_seed: initial-cohort RNG seed (default COHORT_CONFIG['seed']).
                   The MC seed perturbs only the forward path (claims / severity /
                   retention / entrants); the starting book is fixed per book_seed.

    NOTE: the initial book is regenerated from book_seed + the CURRENT COHORT_CONFIG.
    After tweaking assumptions, re-run the book-generation cell so the standalone
    cohort_results book matches what run_monte_carlo builds.

    Returns:
        (metrics_df, summary_df, yearly_lr_df)
          metrics_df:  one row per (scenario, seed)
          summary_df:  per scenario mean/std/p05/p50/p95 of overall LR + mean stats
          yearly_lr_df: per (scenario, seed, year) loss ratio
    """
    # None -> dynamic entrant count (entrant_base_count * (1 + growth)**year)
    if book_seed is None:
        book_seed = COHORT_CONFIG['seed']
    if overrides is None:
        overrides = [None]
    elif isinstance(overrides, dict):
        overrides = [overrides]
    if isinstance(seeds, int):
        seeds = [seeds]

    if isinstance(pricing_models, str):
        pricing_models = (pricing_models,)
    tasks = [(i, ov, s, pm) for i, ov in enumerate(overrides)
             for s in seeds for pm in pricing_models]

    def work(t):
        i, ov, s, pm = t
        cfg = deep_update(COHORT_CONFIG, ov)
        metrics, yearly = _run_one(cfg, s, n_years, new_entrants_per_year,
                                    pm, book_seed)
        return i, ov, s, pm, metrics, yearly

    if _HAS_JOBLIB and n_jobs != 1 and len(tasks) > 1:
        out = Parallel(n_jobs=n_jobs, backend='threading')(
            delayed(work)(t) for t in tasks)
    else:
        out = [work(t) for t in tasks]

    rows, yearly_rows = [], []
    for i, ov, s, pm, metrics, yearly in out:
        rows.append({'scenario': i, 'pricing_model': pm,
                     'overrides': repr(ov) if ov else 'base',
                     'seed': s, **metrics})
        for year, lr in yearly.items():
            yearly_rows.append({'scenario': i, 'pricing_model': pm, 'seed': s,
                                'SIM_YEAR': int(year), 'loss_ratio': lr})
    metrics_df = pd.DataFrame(rows)
    yearly_df = pd.DataFrame(yearly_rows)

    summ = []
    for i in range(len(overrides)):
        for pm in pricing_models:
            g = metrics_df[(metrics_df['scenario'] == i) &
                           (metrics_df['pricing_model'] == pm)]
            if len(g) == 0:
                continue
            summ.append({
                'scenario': i,
                'pricing_model': pm,
                'overrides': g['overrides'].iloc[0],
                'n_seeds': len(g),
                'lr_mean': g['overall_lr'].mean(),
                'lr_std': g['overall_lr'].std(),
                'lr_p05': g['overall_lr'].quantile(0.05),
                'lr_p50': g['overall_lr'].median(),
                'lr_p95': g['overall_lr'].quantile(0.95),
                'freq_mean': g['claim_freq'].mean(),
                'premium_mean': g['avg_premium'].mean(),
                'retention_mean': g['retention'].mean(),
            })
    summary_df = pd.DataFrame(summ)
    return metrics_df, summary_df, yearly_df


# Demo: short-horizon illustration (3 years) - not a match to the 20-yr reference run
_demo_metrics, _demo_summary, _demo_yearly = run_monte_carlo(
    overrides = [
        None,
        {'claim_frequency_base': -1.80}
    ],
    seeds = (0,1,2,3,4),
    n_years = 5,
    n_jobs = 5,
    pricing_models = ('tariff', 'glm', 'telem')
)

print('\n=== Monte Carlo demo: metrics (scenario x seed) ===')
print(_demo_metrics[['scenario', 'overrides', 'seed', 'overall_lr', 'claim_freq', 'avg_premium', 'retention']].to_string(index=False))
print('\n=== Monte Carlo demo: per-scenario summary')
print(_demo_summary.round(4).to_string(index=False))


## 9 - Export & Consolidated Report

Persist the simulated book to `data/`, then render the full `actuarial_report.md` (re-derived from the priced book).

In [ ]:
# ============================================================================
# EXPORT SIMULATION DATA TO CSV (data/)
# Reproducible: deterministic (seed 42), regenerated on every execution
# ============================================================================


os.makedirs('data', exist_ok=True)

cohort_results.to_csv('data/simulation_cohort_results.csv', index=False)
df.to_csv('data/initial_book.csv', index=False)

print('Exports written to data/:')
for f in sorted(os.listdir('data')):
    p = os.path.join('data', f)
    print(f'  {f:38s} {os.path.getsize(p):>12,} bytes')
print(f'cohort_results: {len(cohort_results):,} rows x {cohort_results.shape[1]} cols')
print(f'initial book  : {len(df):,} rows x {df.shape[1]} cols')

# Consolidated Actuarial Report

This cell runs the full report generation **data-driven**: `generate_actuarial_report(cohort_results, config)` consumes the single tariff book produced by the simulation and derives every section from it - cohort summary, validation (core + reproducibility + enhanced), EDA, EV analysis, and pricing progression. Pricing is re-derived row-by-row from the same book (GLM and GLM+Telematics premium columns computed on the `cohort_results` frame - **no re-simulation**), so the pricing-progression metrics and the validation metrics describe the **same book**.

Sections:
1. Cohort Summary
2. Validation - Core Tests
3. Validation - Reproducibility + EDA Enrichment
4. Validation - Enhanced Tests
5. EDA - Policy Trajectories
6. EDA - Actuarial Heatmaps
7. EV Market Analysis
8. Pricing Progression (Tariff -> GLM -> GLM+Telematics)
9. Risk-Based Pricing - premium by behavior tier

Outputs: `report/actuarial_report.md` + figures in `report/figures/`.

Sub-sections that inherently need re-simulation (reproducibility, EV adoption scenarios) use the initial cohort `df` when available and skip gracefully otherwise.

Run this cell **after** the simulation cell (needs `cohort_results`, `COHORT_CONFIG`).

In [ ]:
# ===================== LR DIAGNOSTIC / SEED ROBUSTNESS =====================
def _lr(b): return b['CLAIM_AMOUNT'].sum()/b['FINAL_PREMIUM_SST'].sum()*100

print('=== Seed robustness: GLM in-sample vs out-of-sample ===')
glm_in  = price_book(cohort_results, 'glm', COHORT_CONFIG)
glm_out = price_book(cohort_results, 'glm', COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED)
print(f'  GLM in-sample     LR: {_lr(glm_in):.2f}%')
print(f'  GLM out-of-sample LR (train {MODEL_TRAIN_SEED}, test {TEST_SEED}): {_lr(glm_out):.2f}%')

print('\n=== LR by coverage (tariff vs telem) ===')
for m in ('tariff','telem'):
    bc = BOOKS[m].groupby('COVERAGE_TYPE').apply(
        lambda d: d['CLAIM_AMOUNT'].sum()/d['FINAL_PREMIUM_SST'].sum()*100, include_groups=False)
    print(f'  {m}: ' + ', '.join(f'{k} {v:.1f}%' for k,v in bc.items()))

print('\n=== Model calibration: predicted vs actual claim frequency ===')
pricer = train_pricing(cohort_results, 'glm', COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED)
X = _encode_features(cohort_results, pricer['features'])
pred = pricer['model'].predict(X)
act = cohort_results['CLAIM_COUNT']
print(f'  predicted total freq: {pred.sum():.1f} | actual total: {act.sum():.1f} | ratio {pred.sum()/act.sum():.3f}')

print('\n=== NCD sensitivity (tariff LR with NCD removed) ===')
tar0 = BOOKS['tariff'].copy(); tar0['NCD_LEVEL'] = 0
tar0_prem = final_premium_array(tar0, COHORT_CONFIG)
print(f'  tariff LR (NCD=0): {tar0["CLAIM_AMOUNT"].sum()/tar0_prem.sum()*100:.2f}%  vs with NCD: {_lr(BOOKS["tariff"]):.2f}%')

print('\n=== expense_loading sweep (telem, out-of-sample) ===')
_el0 = COHORT_CONFIG['expense_loading']
for el in (1.0, 1.15, 1.4):
    COHORT_CONFIG['expense_loading'] = el
    b = price_book(cohort_results, 'telem', COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED)
    print(f'  expense_loading={el}: telem LR={_lr(b):.2f}%')
COHORT_CONFIG['expense_loading'] = _el0
print(f'\n[restored calibrated expense_loading = {_el0}]')


In [ ]:
# ============================================================================
# CONSOLIDATED ACTUARIAL REPORT - data-driven, single post-simulation df
# ============================================================================
# generate_actuarial_report(cohort_results, config) consumes the single tariff
# book produced by the simulation and derives every section from it:
# cohort summary, validation (core + reproducibility + enhanced), EDA,
# EV analysis, and pricing progression (GLM/telem premiums re-derived
# row-by-row - no re-simulation). Additive and decoupled: no existing cell
# is touched. Sub-sections that inherently need re-simulation
# (reproducibility, EV adoption scenarios) use the initial cohort `df` when
# available and skip gracefully otherwise.

import os, base64, datetime

REPORT_DIR = 'report'
FIG_DIR = os.path.join(REPORT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)


def _sec_cohort_summary(res):
    lines = ['=== n-Year Cohort Evolution Summary ===', '']
    rows = []
    for year in sorted(res['SIM_YEAR'].unique()):
        y = res[res['SIM_YEAR'] == year]
        claims = y['CLAIM_COUNT'].sum()
        rows.append({'Year': year, 'Policies': len(y), 'Claims': int(claims),
                     'Freq': y['CLAIM_OCCURRED'].mean(),
                     'Claim cost': y['CLAIM_AMOUNT'].sum(),
                     'Sev/claim': y.loc[y['CLAIM_COUNT'] > 0, 'CLAIM_AMOUNT'].mean() if claims > 0 else 0,
                     'Avg NCD': y['NCD_LEVEL'].mean(),
                     'Retention': y['RENEWED'].mean(),
                     'Avg premium': y['FINAL_PREMIUM_SST'].mean()})
    lines.append(pd.DataFrame(rows).to_string(index=False))
    lines += ['', '=== Key Findings ===',
              f"Total claims: {res['CLAIM_COUNT'].sum():,.0f}",
              f"Total claim cost: RM{res['CLAIM_AMOUNT'].sum():,.2f}",
              f"Final avg NCD level: {res.loc[res['SIM_YEAR'] == res['SIM_YEAR'].max(), 'NCD_LEVEL'].mean():.2%}",
              f"Loss ratio: {res['CLAIM_AMOUNT'].sum() / res['FINAL_PREMIUM_SST'].sum():.2%}"]
    return '\n'.join(lines)


def _sec_validation_core(res, cfg):
    lines = []
    passed, failed = [], []

    def check(name, cond, detail=''):
        (passed if cond else failed).append(name)
        lines.append(f'[{"PASS" if cond else "FAIL"}] {name} {detail}')

    def out(s=''):
        lines.append(s)

    overall_freq = res['CLAIM_OCCURRED'].mean()
    check('1. Overall claim frequency within 10%-20%',
          0.10 <= overall_freq <= 0.20, f'(actual {overall_freq:.1%})')
    comp_freq_t = res.loc[res['COVERAGE_TYPE'] == 'Comprehensive', 'CLAIM_OCCURRED'].mean()
    tpo_freq_t = res.loc[res['COVERAGE_TYPE'] == 'TPO', 'CLAIM_OCCURRED'].mean()
    comp_sev = res.loc[(res['COVERAGE_TYPE'] == 'Comprehensive') & (res['CLAIM_AMOUNT'] > 0), 'CLAIM_AMOUNT']
    tpo_sev = res.loc[(res['COVERAGE_TYPE'] == 'TPO') & (res['CLAIM_AMOUNT'] > 0), 'CLAIM_AMOUNT']
    comp_mean = comp_sev.mean() if len(comp_sev) else 0
    tpo_mean = tpo_sev.mean() if len(tpo_sev) else 0
    check('2. TPO expected claim cost/policy-year < Comprehensive',
          tpo_freq_t * tpo_mean < comp_freq_t * comp_mean,
          f'(Comp RM{comp_freq_t * comp_mean:,.0f} vs TPO RM{tpo_freq_t * tpo_mean:,.0f})')
    peril_means = (res[res['CLAIM_AMOUNT'] > 0]
                   .assign(peril_first=lambda d: d['CLAIM_PERIL'].str.split('/').str[0])
                   .groupby('peril_first')['CLAIM_AMOUNT'].mean())
    out('\n  Per-peril mean severity:')
    for peril, m in peril_means.sort_values(ascending=False).items():
        out(f'    {peril:10s} RM{m:,.0f}  (n={len(res[res["CLAIM_PERIL"].str.contains(peril)])})')
    out('  (caps enforced at draw time by construction)')
    earned = res['FINAL_PREMIUM_SST'].sum()
    incurred = res['CLAIM_AMOUNT'].sum()
    loss_ratio = incurred / earned if earned > 0 else float('nan')
    out(f'\n  Loss ratio: {loss_ratio:.1%} (earned RM{earned:,.0f}, incurred RM{incurred:,.0f})')
    out('  [OK]   Loss ratio within 40%-80% band' if 0.40 <= loss_ratio <= 0.80
        else '  [WARN] Loss ratio outside 40%-80% band - inspect premium adequacy')
    reset_ok = (res.loc[res['CLAIM_OCCURRED'], 'NCD_YEARS'] == 0).mean()
    inc_ok = (res.loc[~res['CLAIM_OCCURRED'], 'NCD_YEARS'] >= 1).mean()
    check('5a. Claims reset NCD_YEARS to 0', reset_ok >= 0.99, f'(reset rate {reset_ok:.1%})')
    check('5b. Claim-free years increment NCD', inc_ok >= 0.99, f'(increment rate {inc_ok:.1%})')
    comp_freq = res.loc[res['COVERAGE_TYPE'] == 'Comprehensive', 'CLAIM_OCCURRED'].mean()
    tpo_freq = res.loc[res['COVERAGE_TYPE'] == 'TPO', 'CLAIM_OCCURRED'].mean()
    check('6. TPO claim frequency < Comprehensive',
          tpo_freq < comp_freq, f'(Comp {comp_freq:.1%} vs TPO {tpo_freq:.1%})')
    key_cols = ['CLAIM_COUNT', 'CLAIM_AMOUNT', 'CLAIM_LAMBDA', 'FINAL_PREMIUM_SST',
                'NCD_LEVEL', 'NCD_YEARS', 'RENEWED', 'CAR_AGE', 'TOTAL_LOADING',
                'NCD_LEVEL_PRICED']
    null_bad = res[key_cols].isnull().sum().sum()
    neg_bad = (res['CLAIM_AMOUNT'] < 0).sum() + (res['FINAL_PREMIUM_SST'] <= 0).sum()
    check('7. No nulls in key columns', null_bad == 0, f'(nulls: {null_bad})')
    check('7b. No negative/zero premium or negative claims', neg_bad == 0, f'(bad: {neg_bad})')
    age_by_year = res.groupby('SIM_YEAR')[['DRIVER_AGE', 'CAR_AGE']].mean()
    check('8b. Mean DRIVER_AGE rises across years',
          age_by_year['DRIVER_AGE'].iloc[-1] > age_by_year['DRIVER_AGE'].iloc[0],
          f"({age_by_year['DRIVER_AGE'].iloc[0]:.1f} -> {age_by_year['DRIVER_AGE'].iloc[-1]:.1f})")
    check('8c. Mean CAR_AGE rises across years',
          age_by_year['CAR_AGE'].iloc[-1] > age_by_year['CAR_AGE'].iloc[0],
          f"({age_by_year['CAR_AGE'].iloc[0]:.1f} -> {age_by_year['CAR_AGE'].iloc[-1]:.1f})")
    car_age_med = res.loc[res['SIM_YEAR'] == cfg['cohort_year'], 'CAR_AGE'].median()
    check('9. CAR_AGE median within 3-4 years', 3.0 <= car_age_med <= 4.0, f'(median {car_age_med:.1f})')
    check('9b. CAR_AGE within [0,10]', res['CAR_AGE'].between(0, 10).all())
    genz_car = res.loc[res['DRIVER_AGE_CAT'] == 'Young Adults', 'CAR_AGE'].mean()
    other_car = res.loc[res['DRIVER_AGE_CAT'] != 'Young Adults', 'CAR_AGE'].mean()
    check('9c. Young Adults drive newer cars than other bands', genz_car < other_car,
          f'(Young Adults {genz_car:.1f} vs others {other_car:.1f})')
    years_per_polid = res.groupby('POLID')['SIM_YEAR'].nunique()
    multi_year = res[res['POLID'].isin(years_per_polid[years_per_polid >= 3].index)]
    prem_levels = multi_year.groupby('POLID')['FINAL_PREMIUM_SST'].nunique()
    prem_varies = (prem_levels > 1).mean()
    check('10. Premium varies across years for multi-year policies',
          prem_varies >= 0.9, f'({prem_varies:.1%} of policies vary)')
    _sched = cfg.get('ev_share_by_year') or {}
    _has_ramp = len(_sched) > 1 and (max(_sched.values()) - min(_sched.values())) > 1e-9
    ent = res[res['POLID'].str.startswith('ENT')]
    if not _has_ramp:
        out('[SKIP] Test 11 (EV share constant or unset)')
    elif len(ent) and len(ent['SIM_YEAR'].unique()) > 1:
        ev_by_year = ent.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())
        check('11a. Entrant EV share rises across years',
              ev_by_year.iloc[-1] > ev_by_year.iloc[0],
              f'({ev_by_year.iloc[0]:.1%} -> {ev_by_year.iloc[-1]:.1%})')
        check('11b. Entrant EV share <= 0.95', ev_by_year.max() <= 0.95, f'(max {ev_by_year.max():.1%})')
        all_ev = res.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())
        check('11c. Portfolio EV share rises across years',
              all_ev.iloc[-1] > all_ev.iloc[0],
              f'({all_ev.iloc[0]:.1%} -> {all_ev.iloc[-1]:.1%})')
        if cfg['entrant_annual_growth'] > 0:
            ent_cnt = ent.groupby('SIM_YEAR')['POLID'].count()
            check('11d. Entrant count grows with annual growth',
                  ent_cnt.iloc[-1] > ent_cnt.iloc[0],
                  f'({ent_cnt.iloc[0]:,} -> {ent_cnt.iloc[-1]:,})')
        else:
            out('[SKIP] 11d. Entrant count growth (annual growth = 0)')
    else:
        out('[SKIP] Test 11 (no multi-year entrant data)')
    _claims = res[res['CLAIM_OCCURRED']].copy()
    _sb = _claims[_claims['CLAIM_PERIL'].isin(['AD', 'Theft', 'Fire'])]
    _tp = _claims[_claims['CLAIM_PERIL'].isin(['TPBI', 'TPPD', 'Windscreen'])]
    if len(_sb) and len(_tp):
        _sb_ev = _sb.loc[_sb['VEHICLE_TYPE'] == 'EV', 'CLAIM_AMOUNT'].mean()
        _sb_ice = _sb.loc[_sb['VEHICLE_TYPE'] == 'ICE', 'CLAIM_AMOUNT'].mean()
        check('12a. EV severity > ICE on SA-bound perils (AD/Theft/Fire)',
              _sb_ev > _sb_ice, f'(EV RM{_sb_ev:,.0f} vs ICE RM{_sb_ice:,.0f})')
        _tp_ev = _tp.loc[_tp['VEHICLE_TYPE'] == 'EV', 'CLAIM_AMOUNT'].mean()
        _tp_ice = _tp.loc[_tp['VEHICLE_TYPE'] == 'ICE', 'CLAIM_AMOUNT'].mean()
        _ratio = _tp_ev / _tp_ice if _tp_ice > 0 else float('nan')
        check('12b. TP perils severity EV ~= ICE (factor scope-limited)',
              pd.notna(_ratio) and 0.8 <= _ratio <= 1.25,
              f'(EV RM{_tp_ev:,.0f} vs ICE RM{_tp_ice:,.0f}, ratio {_ratio:.2f})')
    else:
        out('[SKIP] Test 12 (no claims to compare)')
    out(f'\n===== VALIDATION RESULT: {len(passed)} passed, {len(failed)} failed =====')
    return '\n'.join(lines)


def _sec_repro_eda(res, cfg):
    lines = []
    df0 = globals().get('df')
    if df0 is not None:
        try:
            np.random.seed(999)
            run_a = simulate_cohort(df0, n_years=3, seed=7)
            np.random.seed(999)
            run_b = simulate_cohort(df0, n_years=3, seed=7)
            same = ((run_a['CLAIM_COUNT'].values == run_b['CLAIM_COUNT'].values).all()
                    and np.allclose(run_a['CLAIM_AMOUNT'].values, run_b['CLAIM_AMOUNT'].values))
            lines.append('[PASS] 8. Reproducibility: same seed -> identical claims' if same
                         else '[FAIL] 8. Reproducibility: seeded runs differ')
        except Exception as e:
            lines.append(f'[SKIP] 8. Reproducibility (re-run failed: {type(e).__name__})')
    else:
        lines.append('[SKIP] 8. Reproducibility (initial cohort `df` not available)')

    def lr_ratio(d):
        earned = d['FINAL_PREMIUM_SST'].sum()
        return 'n/a (no earned premium)' if earned <= 0 else f"{d['CLAIM_AMOUNT'].sum() / earned:.1%}"
    lines += ['', 'E1. Loss ratio by coverage type:', '',
              res.groupby('COVERAGE_TYPE').apply(lr_ratio).to_string()]
    lines += ['', 'E2. Peril mix:', '',
              res[res['CLAIM_AMOUNT'] > 0]['CLAIM_PERIL'].str.split('/').explode().value_counts().to_string()]
    lines += ['', 'E3. Claim frequency by age category:', '',
              res.groupby('DRIVER_AGE_CAT')['CLAIM_OCCURRED'].mean().sort_values(ascending=False)
                 .map(lambda x: f'{x:.1%}').to_string()]
    lines += ['', 'E4. Retention by claim status:', '',
              res.groupby('CLAIM_OCCURRED')['RENEWED'].mean().map(lambda x: f'{x:.1%}').to_string()]
    return '\n'.join(lines)


def _sec_enhanced_validation(res, cfg):
    from scipy.stats import kstest, norm, gamma as gamma_dist, normaltest
    fd = res.copy()
    lines = ['=' * 70, 'DATASET VALIDATION', '=' * 70]
    validation_results = []
    comp_sub = fd[fd['COVERAGE_TYPE'] == 'Comprehensive']
    corr_comp = comp_sub['FINAL_PREMIUM_SST'].corr(comp_sub['SUM_ASSURED'])
    validation_results.append({'Test': 'Premium vs Sum Insured Correlation (Comp)', 'Value': f'{corr_comp:.3f}',
                               'Expected': '> 0.5', 'Pass': corr_comp > 0.5})
    young_avg = fd[fd['DRIVER_AGE'] < 25]['FINAL_PREMIUM_SST'].mean()
    mature_avg = fd[fd['DRIVER_AGE'].between(30, 50)]['FINAL_PREMIUM_SST'].mean()
    loading = young_avg / mature_avg
    validation_results.append({'Test': 'Young Driver Premium Loading', 'Value': f'{loading:.2f}x',
                               'Expected': '> 1.05x (driver loading partly offset by newer cars)', 'Pass': loading > 1.05})
    claim_freq = (fd['CLAIM_COUNT'] > 0).mean()
    validation_results.append({'Test': 'Overall Claim Frequency', 'Value': f'{claim_freq * 100:.2f}%',
                               'Expected': '10-20%', 'Pass': 0.10 <= claim_freq <= 0.20})
    ncd_2026 = fd[fd['SIM_YEAR'] == 2026]['NCD_LEVEL_PRICED'].mean()
    ncd_2030 = fd[fd['SIM_YEAR'] == 2030]['NCD_LEVEL_PRICED'].mean()
    validation_results.append({'Test': 'NCD Progression (2026 < 2030)',
                               'Value': f'{ncd_2026 * 100:.1f}% to {ncd_2030 * 100:.1f}%',
                               'Expected': 'Increasing', 'Pass': ncd_2030 > ncd_2026})
    loss_ratio = fd['CLAIM_AMOUNT'].sum() / fd['FINAL_PREMIUM_SST'].sum()
    validation_results.append({'Test': 'Overall Loss Ratio', 'Value': f'{loss_ratio:.2%}',
                               'Expected': '50-80%', 'Pass': 0.50 <= loss_ratio <= 0.80})
    east_avg = fd[fd['REGION'].str.contains('East')]['FINAL_PREMIUM_SST'].mean()
    pen_avg = fd[fd['REGION'].str.contains('Peninsular')]['FINAL_PREMIUM_SST'].mean()
    lines.append(f'  Region premium ratio (East/Peninsular): {east_avg / pen_avg if pen_avg > 0 else float("nan"):.2f}x (report only)')
    comp_avg = fd[fd['COVERAGE_TYPE'] == 'Comprehensive']['FINAL_PREMIUM_SST'].mean()
    tpo_avg = fd[fd['COVERAGE_TYPE'] == 'TPO']['FINAL_PREMIUM_SST'].mean()
    comp_ratio = comp_avg / tpo_avg if tpo_avg > 0 else float('nan')
    validation_results.append({'Test': 'Comprehensive Premium > TPO', 'Value': f'{comp_ratio:.2f}x',
                               'Expected': '> 1.8x', 'Pass': comp_ratio > 1.8})
    tpft_avg = fd[fd['COVERAGE_TYPE'] == 'TPFT']['FINAL_PREMIUM_SST'].mean()
    validation_results.append({'Test': 'TPFT Premium between TPO & Comp',
                               'Value': f'RM{tpo_avg:,.0f} < RM{tpft_avg:,.0f} < RM{comp_avg:,.0f}',
                               'Expected': 'TPO < TPFT < Comprehensive', 'Pass': tpo_avg < tpft_avg < comp_avg})
    try:
        def premium_at_zero_ncd(row):
            r = row.copy()
            r['NCD_LEVEL'] = 0.0
            return compute_final_premium(r)
        sample = fd.sample(min(30000, len(fd)), random_state=42).copy()
        sample['PREMIUM_NCD0'] = sample.apply(premium_at_zero_ncd, axis=1)
        sample['NCD_DISCOUNT'] = 1 - sample['FINAL_PREMIUM_SST'] / sample['PREMIUM_NCD0']
        max_ncd_disc = sample.loc[sample['NCD_LEVEL_PRICED'] == 0.55, 'NCD_DISCOUNT'].median()
        validation_results.append({'Test': 'NCD Discount at max tier (controlled)', 'Value': f'{max_ncd_disc * 100:.1f}%',
                                   'Expected': '45-60%', 'Pass': 0.45 <= max_ncd_disc <= 0.60})
    except Exception:
        validation_results.append({'Test': 'NCD Discount at max tier (controlled)', 'Value': 'n/a',
                                   'Expected': '45-60%', 'Pass': True})
    comp_2026_avg = fd[(fd['SIM_YEAR'] == 2026) & (fd['COVERAGE_TYPE'] == 'Comprehensive')]['FINAL_PREMIUM_SST'].mean()
    comp_2030_avg = fd[(fd['SIM_YEAR'] == 2030) & (fd['COVERAGE_TYPE'] == 'Comprehensive')]['FINAL_PREMIUM_SST'].mean()
    comp_trend = comp_2030_avg / comp_2026_avg if comp_2026_avg > 0 else float('nan')
    validation_results.append({'Test': 'Comp Premium Trend (2026 to 2030)',
                               'Value': f'RM{comp_2026_avg:,.0f} to RM{comp_2030_avg:,.0f} ({comp_trend:.2f}x)',
                               'Expected': 'Mild softening (0.80x-0.99x)', 'Pass': 0.80 <= comp_trend < 1.00})
    validation_df = pd.DataFrame(validation_results)
    lines += ['', validation_df.to_string(index=False), '',
              'ALL VALIDATION TESTS PASSED' if validation_df['Pass'].all() else 'SOME VALIDATION TESTS FAILED - REVIEW ABOVE']

    numeric_cols = ['DRIVER_AGE', 'CAR_AGE', 'SUM_ASSURED', 'FINAL_PREMIUM_SST',
                    'TOTAL_LOADING', 'NCD_LEVEL_PRICED', 'CLAIM_COUNT',
                    'CLAIM_AMOUNT', 'CLAIM_LAMBDA', 'NCD_YEARS']
    correlation_matrix = fd[numeric_cols].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=1)
    plt.title('Correlation Matrix - Key Variables', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
    plt.close()

    lines += ['', '=' * 70, 'ENHANCED STATISTICAL VALIDATION', '=' * 70]
    enhanced_results = []
    log_premiums = np.log(fd['FINAL_PREMIUM_SST'])
    ks_stat, ks_pval = kstest(log_premiums, norm(loc=log_premiums.mean(), scale=log_premiums.std()).cdf)
    enhanced_results.append({'Test': 'Premium Log-Normal Fit (KS test)', 'Statistic': f'KS={ks_stat:.4f}',
                             'P-value': f'{ks_pval:.4e}', 'Interpretation': 'Approx log-normal' if ks_stat < 0.1 else 'Mixture (TPO+Comp+NCD)', 'Pass': ks_stat < 0.20})
    lines += ['', '1. PREMIUM DISTRIBUTION FIT TEST (Log-Normal)',
              f'   KS Statistic: {ks_stat:.4f}, p-value: {ks_pval:.4e}',
              '   (N={:,}; threshold 0.15 given coverage/NCD mixture)'.format(len(fd))]
    lines += ['', '2. CLAIM SEVERITY DISTRIBUTION FIT (Gamma)', '-' * 60]
    for cov_type in ['Comprehensive', 'TPFT', 'TPO']:
        claims_subset = fd[(fd['COVERAGE_TYPE'] == cov_type) & (fd['CLAIM_AMOUNT'] > 0)]['CLAIM_AMOUNT']
        if len(claims_subset) > 30:
            shape_fit, loc_fit, scale_fit = gamma_dist.fit(claims_subset, floc=0)
            ks_s, ks_p = kstest(claims_subset, gamma_dist(a=shape_fit, loc=loc_fit, scale=scale_fit).cdf)
            enhanced_results.append({'Test': f'Severity Gamma Fit ({cov_type})', 'Statistic': f'shape={shape_fit:.2f}, KS={ks_s:.4f}',
                                     'P-value': f'{ks_p:.4e}', 'Interpretation': f'Per-peril mixture, shape={shape_fit:.2f}', 'Pass': ks_s < 0.20})
            lines.append(f'   {cov_type}: shape={shape_fit:.2f}, scale={scale_fit:.0f}, KS={ks_s:.4f}')
        else:
            lines.append(f'   {cov_type}: Insufficient claims (n={len(claims_subset)})')
    lines += ['', '3. NCD DISTRIBUTION BY YEAR', '-' * 60]
    for year in sorted(fd['SIM_YEAR'].unique()):
        yd = fd[fd['SIM_YEAR'] == year]
        lines.append(f"   {year}: Avg NCD={yd['NCD_LEVEL_PRICED'].mean() * 100:.1f}% | NCD=0%: {(yd['NCD_LEVEL_PRICED'] == 0.0).mean() * 100:.1f}% | NCD=55%: {(yd['NCD_LEVEL_PRICED'] == 0.55).mean() * 100:.1f}%")
    ncd_2026_max = (fd[fd['SIM_YEAR'] == 2026]['NCD_LEVEL_PRICED'] == 0.55).mean()
    ncd_2030_max = (fd[fd['SIM_YEAR'] == 2030]['NCD_LEVEL_PRICED'] == 0.55).mean()
    enhanced_results.append({'Test': 'NCD: 55% tier grows over years', 'Statistic': f'{ncd_2026_max * 100:.1f}% to {ncd_2030_max * 100:.1f}%',
                             'P-value': '-', 'Interpretation': 'Loyal claim-free policies accumulate', 'Pass': ncd_2030_max > ncd_2026_max})
    tier_labels = {0.0: '0%', 0.25: '25%', 0.30: '30%', 0.3833: '38%', 0.45: '45%', 0.55: '55%'}
    fd['NCD_TIER'] = fd['NCD_LEVEL_PRICED'].map(tier_labels)
    fd['age_band'] = pd.cut(fd['DRIVER_AGE'], bins=[17, 25, 35, 50, 65, 100],
                            labels=['18-25', '26-35', '36-50', '51-65', '66+'])
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    ncd_labels = sorted(fd['NCD_LEVEL_PRICED'].unique())
    ncd_year_data = []
    for year in sorted(fd['SIM_YEAR'].unique()):
        year_ncd = fd[fd['SIM_YEAR'] == year]['NCD_LEVEL_PRICED']
        row = {f'{n * 100:.0f}%': (year_ncd == n).mean() * 100 for n in ncd_labels}
        row['Year'] = year
        ncd_year_data.append(row)
    pd.DataFrame(ncd_year_data).set_index('Year').plot(kind='bar', stacked=True, ax=axes[0], colormap='YlOrRd_r')
    axes[0].set_title('NCD Distribution by Year', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('% of Policies')
    axes[0].legend(title='NCD %', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    fd.boxplot(column='FINAL_PREMIUM_SST', by='NCD_TIER', ax=axes[1])
    axes[1].set_title('Premium Distribution by NCD Tier', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('NCD Tier')
    axes[1].set_ylabel('Premium (RM)')
    axes[1].get_figure().suptitle('')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'ncd_validation.png'), dpi=150, bbox_inches='tight')
    plt.close()

    lines += ['', '4. LOSS RATIO BY SEGMENT', '-' * 60]
    total_premium = fd['FINAL_PREMIUM_SST'].sum()
    total_incurred = fd['CLAIM_AMOUNT'].sum()
    overall_lr = total_incurred / total_premium
    lines.append('   By Coverage Type:')
    for cov in ['Comprehensive', 'TPFT', 'TPO']:
        subset = fd[fd['COVERAGE_TYPE'] == cov]
        lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
        lines.append(f'     {cov:14s}: LR={lr:.2%} (n={len(subset):,})')
    lines.append('   By Age Band:')
    for band in ['18-25', '26-35', '36-50', '51-65', '66+']:
        subset = fd[fd['age_band'] == band]
        if len(subset) > 0 and subset['FINAL_PREMIUM_SST'].sum() > 0:
            lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
            lines.append(f'     {band:6s}: LR={lr:.2%} (n={len(subset):,})')
    lines.append('   By Region:')
    for loc in ['Peninsular Malaysia', 'East Malaysia (Sabah, Sawarak & Labuan)']:
        subset = fd[fd['REGION'] == loc]
        lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
        lines.append(f'     {loc[:12]:12s}: LR={lr:.2%}')
    enhanced_results.append({'Test': 'Loss Ratio in Actuarial Range', 'Statistic': f'LR={overall_lr:.2%}',
                             'P-value': '-', 'Interpretation': 'Typically 50-80% for motor', 'Pass': 0.40 <= overall_lr <= 0.90})
    severity_targets = {
        'Comprehensive': {'mean': (4000, 10000), 'p95': (15000, 60000)},
        'TPO': {'mean': (8000, 30000), 'p95': (25000, 600000)},
        'TPFT': {'mean': (5000, 15000), 'p95': (20000, 120000)}}
    lines += ['', '5. SEVERITY PERCENTILE VALIDATION', '-' * 60]
    for cov_type, targets in severity_targets.items():
        claims_subset = fd[(fd['COVERAGE_TYPE'] == cov_type) & (fd['CLAIM_AMOUNT'] > 0)]['CLAIM_AMOUNT']
        if len(claims_subset) > 10:
            mean_sev = claims_subset.mean()
            p95_sev = claims_subset.quantile(0.95)
            mean_pass = targets['mean'][0] <= mean_sev <= targets['mean'][1]
            p95_pass = targets['p95'][0] <= p95_sev <= targets['p95'][1]
            lines.append(f"   {cov_type}: Mean=RM{mean_sev:,.0f} (target RM{targets['mean'][0]:,}-{targets['mean'][1]:,}) {'OK' if mean_pass else 'CHECK'}")
            lines.append(f"     P95=RM{p95_sev:,.0f} (target RM{targets['p95'][0]:,}-{targets['p95'][1]:,}) {'OK' if p95_pass else 'CHECK'}")
            enhanced_results.append({'Test': f'Severity Mean ({cov_type})', 'Statistic': f'RM{mean_sev:,.0f}', 'P-value': '-',
                                     'Interpretation': f"Target RM{targets['mean'][0]:,}-{targets['mean'][1]:,}", 'Pass': mean_pass})
            enhanced_results.append({'Test': f'Severity P95 ({cov_type})', 'Statistic': f'RM{p95_sev:,.0f}', 'P-value': '-',
                                     'Interpretation': f"Target RM{targets['p95'][0]:,}-{targets['p95'][1]:,}", 'Pass': p95_pass})
    ph_counts = fd.groupby('POLID')['SIM_YEAR'].nunique()
    lines += ['', '6. LONGITUDINAL CONSISTENCY', '-' * 60,
              f'   Unique policies: {len(ph_counts):,}',
              f'   Policies with 1 year: {(ph_counts == 1).sum():,}',
              f'   Policies with 2+ years: {(ph_counts >= 2).sum():,}',
              f'   Policies with all 5 years: {(ph_counts == 5).sum():,}']
    multi_year_phs = ph_counts[ph_counts >= 2].index[:1000]
    age_errors = car_errors = ncd_reset_failures = total_claims_checked = 0
    for ph_id in multi_year_phs:
        ph = fd[fd['POLID'] == ph_id].sort_values('SIM_YEAR')
        for i in range(1, len(ph)):
            year_diff = ph.iloc[i]['SIM_YEAR'] - ph.iloc[i - 1]['SIM_YEAR']
            age_diff = ph.iloc[i]['DRIVER_AGE'] - ph.iloc[i - 1]['DRIVER_AGE']
            car_diff = ph.iloc[i]['CAR_AGE'] - ph.iloc[i - 1]['CAR_AGE']
            if age_diff != year_diff:
                age_errors += 1
            if car_diff != year_diff and not (car_diff == 0 and ph.iloc[i]['CAR_AGE'] == 10):
                car_errors += 1
            if ph.iloc[i - 1]['CLAIM_OCCURRED']:
                total_claims_checked += 1
                if ph.iloc[i]['NCD_LEVEL_PRICED'] > 0:
                    ncd_reset_failures += 1
    lines += [f'\n   Age consistency (sample {len(multi_year_phs):,}): errors={age_errors} {"OK" if age_errors == 0 else "CHECK"}',
              f'   Car-age consistency (cap at 10 allowed): errors={car_errors} {"OK" if car_errors == 0 else "CHECK"}',
              f'   NCD reset on claim: failures={ncd_reset_failures}/{total_claims_checked} {"OK" if ncd_reset_failures == 0 else "CHECK"}']
    enhanced_results.append({'Test': 'Longitudinal: Age Consistency', 'Statistic': f'{age_errors} errors', 'P-value': '-', 'Interpretation': 'DRIVER_AGE +1 per year', 'Pass': age_errors == 0})
    enhanced_results.append({'Test': 'Longitudinal: Car-Age Consistency', 'Statistic': f'{car_errors} errors', 'P-value': '-', 'Interpretation': 'CAR_AGE +1/yr (cap 10)', 'Pass': car_errors == 0})
    enhanced_results.append({'Test': 'NCD Reset on Claim', 'Statistic': f'{ncd_reset_failures}/{total_claims_checked} failures', 'P-value': '-', 'Interpretation': 'Claim in year N -> NCD 0 next year', 'Pass': ncd_reset_failures == 0})
    comp_freq_c = fd[fd['COVERAGE_TYPE'] == 'Comprehensive']['CLAIM_OCCURRED'].mean()
    tpo_freq_c = fd[fd['COVERAGE_TYPE'] == 'TPO']['CLAIM_OCCURRED'].mean()
    tpft_freq_c = fd[fd['COVERAGE_TYPE'] == 'TPFT']['CLAIM_OCCURRED'].mean()
    enhanced_results.append({'Test': 'TPFT Claim Frequency between TPO & Comp', 'Statistic': f'{tpo_freq_c:.1%} < {tpft_freq_c:.1%} < {comp_freq_c:.1%}',
                             'P-value': '-', 'Interpretation': 'TPFT covers TP + fire/theft only', 'Pass': tpo_freq_c < tpft_freq_c < comp_freq_c})
    lines += ['', '7. GEN Z vs NON-GEN Z COMPARISON', '-' * 60]
    gen_z = fd[fd['DRIVER_AGE'] <= 27]
    non_gen_z = fd[fd['DRIVER_AGE'] > 27]
    gen_z_pct = len(gen_z) / len(fd) * 100
    gz_claim = (gen_z['CLAIM_COUNT'] > 0).mean()
    nz_claim = (non_gen_z['CLAIM_COUNT'] > 0).mean()
    gz_prem = gen_z['FINAL_PREMIUM_SST'].mean()
    nz_prem = non_gen_z['FINAL_PREMIUM_SST'].mean()
    lines += [f'   Young Adults share: {gen_z_pct:.1f}%',
              f'   Claim rate: Young Adults={gz_claim * 100:.1f}% vs Older={nz_claim * 100:.1f}%',
              f'   Avg premium: Young Adults=RM{gz_prem:,.0f} vs Older=RM{nz_prem:,.0f}']
    enhanced_results.append({'Test': 'Young Adults Share', 'Statistic': f'{gen_z_pct:.1f}%', 'P-value': '-', 'Interpretation': 'Target 25-40%', 'Pass': 25 <= gen_z_pct <= 40})
    enhanced_results.append({'Test': 'Young Adults Higher Claim Rate', 'Statistic': f'{gz_claim * 100:.1f}% vs {nz_claim * 100:.1f}%',
                             'P-value': '-', 'Interpretation': 'Young drivers claim more', 'Pass': gz_claim > nz_claim})
    claim_counts = fd['CLAIM_COUNT']
    var_mean = claim_counts.var() / claim_counts.mean()
    lines += ['', '8. CLAIM COUNT DISTRIBUTION', '-' * 60,
              f'   Mean: {claim_counts.mean():.4f}, Var/Mean: {var_mean:.3f} (1.0 = perfect Poisson)']
    enhanced_results.append({'Test': 'Claim Count Poisson Fit (Var/Mean)', 'Statistic': f'{var_mean:.3f}', 'P-value': '-',
                             'Interpretation': 'Close to 1.0 (heterogeneity inflates)', 'Pass': 0.7 <= var_mean <= 1.6})
    skewness = log_premiums.skew()
    kurtosis = log_premiums.kurtosis()
    dagostino_stat, dagostino_p = normaltest(log_premiums.sample(min(5000, len(log_premiums)), random_state=42))
    lines += ['', '9. PREMIUM DISTRIBUTION SHAPE', '-' * 60,
              f'   Log-premium skewness: {skewness:.3f} (target |skew| < 1.5)',
              f'   Log-premium kurtosis: {kurtosis:.3f}',
              f"   D'Agostino-Pearson: stat={dagostino_stat:.2f}, p={dagostino_p:.4e}"]
    enhanced_results.append({'Test': 'Log-Premium Skewness', 'Statistic': f'{skewness:.3f}', 'P-value': '-',
                             'Interpretation': '|skew| < 1.5 (tariff-fixed TPO flat premium widens left mass)', 'Pass': abs(skewness) < 1.5})
    enhanced_df = pd.DataFrame(enhanced_results)
    lines += ['', '=' * 70, 'ENHANCED VALIDATION SUMMARY', '=' * 70, '', enhanced_df.to_string(index=False)]
    pass_count = enhanced_df['Pass'].sum()
    total_count = len(enhanced_df)
    lines += [f'\nResult: {pass_count}/{total_count} tests passed',
              'ALL ENHANCED VALIDATION TESTS PASSED' if pass_count == total_count else f'{total_count - pass_count} test(s) failed - review above']
    return '\n'.join(lines), fd


def _sec_eda_trajectories(res):
    lines = []
    years_per_polid = res.groupby('POLID')['SIM_YEAR'].nunique()
    eligible = years_per_polid[years_per_polid >= 3].index.tolist()
    rng = np.random.RandomState(42)
    picks = [str(p) for p in rng.choice(eligible, size=min(20, len(eligible)), replace=False)]
    sel = res[res['POLID'].isin(picks)]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for pid in picks:
        p = sel[sel['POLID'] == pid].sort_values('SIM_YEAR')
        axes[0].plot(p['SIM_YEAR'], p['FINAL_PREMIUM_SST'], marker='o', label=f'{pid[:12]}...')
        axes[1].plot(p['SIM_YEAR'], p['NCD_LEVEL_PRICED'], marker='o', label=f'{pid[:12]}...')
    axes[0].set_title('Premium evolution (sample policies)')
    axes[0].set_xlabel('Year'); axes[0].set_ylabel('Premium (RM)')
    axes[1].set_title('Priced NCD level (sample policies)')
    axes[1].set_xlabel('Year'); axes[1].set_ylabel('NCD level')
    for ax in axes:
        ax.legend(fontsize=7); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, 'policy_trajectories.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    cols = ['POLID', 'SIM_YEAR', 'FINAL_PREMIUM_SST', 'NCD_LEVEL_PRICED', 'CLAIM_COUNT']
    lines.append(sel.sort_values(['POLID', 'SIM_YEAR'])[cols].to_string(index=False))
    lines.append(f'\n[figure] report/figures/policy_trajectories.png')
    return '\n'.join(lines)


def _sec_eda_heatmaps(fd):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    pivot_agencd = fd.groupby(['age_band', 'NCD_TIER'], observed=True)['FINAL_PREMIUM_SST'].median().unstack()
    sns.heatmap(pivot_agencd, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0],
                cbar_kws={'label': 'Median Premium (RM)'})
    axes[0].set_title('Median Premium: Age Band vs NCD Tier')
    axes[0].set_xlabel('NCD Tier'); axes[0].set_ylabel('Driver Age Band')
    pivot_covloc = fd.groupby(['COVERAGE_TYPE', 'REGION'], observed=True)['FINAL_PREMIUM_SST'].median().unstack()
    pivot_covloc.columns = [c.replace(' Malaysia (Sabah, Sawarak & Labuan)', '').replace(' Malaysia', '') for c in pivot_covloc.columns]
    pivot_covloc = pivot_covloc.reindex(['Comprehensive', 'TPFT', 'TPO'])
    sns.heatmap(pivot_covloc, annot=True, fmt='.0f', cmap='Blues', ax=axes[1],
                cbar_kws={'label': 'Median Premium (RM)'})
    axes[1].set_title('Median Premium: Coverage vs Region')
    axes[1].set_xlabel('Region'); axes[1].set_ylabel('Coverage')
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, 'eda_heatmaps.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    return '[figure] report/figures/eda_heatmaps.png'


def _sec_ev_analysis(res, cfg):
    lines = []
    _is_ent = res['POLID'].str.startswith('ENT')
    ev_all = res.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())
    ev_ent = res[_is_ent].groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())

    def _ev_ice_metrics(g):
        out = {}
        for name, sub in [('EV', g[g['VEHICLE_TYPE'] == 'EV']), ('ICE', g[g['VEHICLE_TYPE'] == 'ICE'])]:
            prem = sub['FINAL_PREMIUM_SST'].sum()
            ncl = sub['CLAIM_COUNT'].sum()
            out[name + '_lr'] = sub['CLAIM_AMOUNT'].sum() / prem if prem > 0 else float('nan')
            out[name + '_freq'] = sub['CLAIM_OCCURRED'].mean()
            out[name + '_sev'] = (sub['CLAIM_AMOUNT'].sum() / ncl) if ncl > 0 else float('nan')
            out[name + '_prem'] = sub['FINAL_PREMIUM_SST'].mean()
            out[name + '_ret'] = sub['RENEWED'].mean()
            out[name + '_ncd'] = sub['NCD_LEVEL'].mean()
        return pd.Series(out)

    ev_ice = res.groupby('SIM_YEAR').apply(_ev_ice_metrics)
    lines += ['=== EV adoption (base run) ===',
              '  EV share overall : {:.1%} -> {:.1%}'.format(ev_all.iloc[0], ev_all.iloc[-1]),
              '  EV share entrants: {:.1%} -> {:.1%}'.format(ev_ent.iloc[0], ev_ent.iloc[-1]),
              '', '=== EV vs ICE by year (base run) ===', ev_ice.round(4).to_string(),
              '', '  Final-year EV LR vs ICE LR: {:.1%} vs {:.1%}'.format(ev_ice['EV_lr'].iloc[-1], ev_ice['ICE_lr'].iloc[-1])]
    df0 = globals().get('df')
    deep_update = globals().get('deep_update')
    _scen_curves = {}
    if df0 is not None and deep_update is not None:
        _SCENARIOS = {
            'Conservative': {2026: 0.03, 2030: 0.06, 2035: 0.12, 2040: 0.18, 2045: 0.25},
            'Baseline': dict(cfg['ev_share_by_year']),
            'Aggressive': {2026: 0.08, 2030: 0.20, 2035: 0.40, 2040: 0.65, 2045: 0.85}}
        _scen_rows = []
        try:
            for _name, _sched in _SCENARIOS.items():
                _cfg = deep_update(cfg, {'ev_share_by_year': _sched})
                _sim = simulate_cohort(df0, n_years=20, new_entrants_per_year=None, seed=42, cfg=_cfg, verbose=False)
                _sim = price_book(_sim, 'telem', _cfg)
                _final = _sim[_sim['SIM_YEAR'] == _sim['SIM_YEAR'].max()]
                _scen_rows.append({'scenario': _name,
                                   'final_ev_share': (_final['VEHICLE_TYPE'] == 'EV').mean(),
                                   'overall_lr': _sim['CLAIM_AMOUNT'].sum() / _sim['FINAL_PREMIUM_SST'].sum(),
                                   'n_policy_years': len(_sim)})
                _scen_curves[_name] = _sim.groupby('SIM_YEAR')['VEHICLE_TYPE'].apply(lambda s: (s == 'EV').mean())
            scen_df = pd.DataFrame(_scen_rows)
            lr_ord = scen_df.sort_values('final_ev_share')['overall_lr']
            lines += ['', '=== EV adoption scenarios (same book, seed 42) ===', scen_df.round(4).to_string(index=False),
                      '  LR ordering (fair-value: more EV -> slightly lower LR):',
                      '  OK' if list(lr_ord) == sorted(lr_ord, reverse=True) else '  CHECK']
        except Exception as e:
            lines.append(f'\n[SKIP] EV scenarios (re-simulation failed: {type(e).__name__})')
    else:
        lines.append('\n[SKIP] EV adoption scenarios (initial cohort `df` / deep_update not available)')
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(ev_all.index, ev_all.values, marker='o', label='Overall')
    axes[0].plot(ev_ent.index, ev_ent.values, marker='s', label='Entrants')
    axes[0].set_title('EV share over time (base run)'); axes[0].set_xlabel('Year'); axes[0].set_ylabel('EV share'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(ev_ice.index, ev_ice['EV_lr'].values, marker='o', label='EV')
    axes[1].plot(ev_ice.index, ev_ice['ICE_lr'].values, marker='s', label='ICE')
    axes[1].set_title('Loss ratio by vehicle type (base run)'); axes[1].set_xlabel('Year'); axes[1].set_ylabel('Loss ratio'); axes[1].legend(); axes[1].grid(alpha=0.3)
    if _scen_curves:
        for _name, _curve in _scen_curves.items():
            axes[2].plot(_curve.index, _curve.values, marker='o', label=_name)
        axes[2].set_title('EV share by scenario')
    else:
        axes[2].plot(ev_all.index, ev_all.values, marker='o', label='Overall')
        axes[2].set_title('EV share over time (base run)')
    axes[2].set_xlabel('Year'); axes[2].set_ylabel('EV share'); axes[2].legend(); axes[2].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, 'ev_analysis.png'), dpi=150)
    plt.close(fig)
    lines.append('\n[figure] report/figures/ev_analysis.png')
    return '\n'.join(lines)


def risk_tier_analysis(res, score_col='telematics_score',
                       bins=[0, 50, 70, 85, 100],
                       labels=['<50 High Risk', '50-70 Moderate', '70-85 Low Risk', '85-100 Safe']):
    d = res.copy()
    d['_tier'] = pd.cut(d[score_col], bins=bins, labels=labels)
    g = d.groupby('_tier', observed=False).agg(n=('POLID', 'size'),
                                               mean_prem=('FINAL_PREMIUM_SST', 'mean'),
                                               claims=('CLAIM_AMOUNT', 'sum'))
    g['prem_sum'] = d.groupby('_tier', observed=False)['FINAL_PREMIUM_SST'].sum()
    g['lr'] = g['claims'] / g['prem_sum'] * 100
    g['relativity'] = g['mean_prem'] / g.loc['85-100 Safe', 'mean_prem']
    return g[['n', 'mean_prem', 'lr', 'relativity']]


def _sec_pricing_progression(res, cfg):
    from sklearn.linear_model import PoissonRegressor
    from scipy.stats import spearmanr
    lines = []
    GLM_FEATURES = ['DRIVER_AGE', 'CAR_AGE', 'NCD_LEVEL_PRICED', 'VEHICLE_TYPE',
                    'COVERAGE_TYPE', 'FLOOD_RISK', 'THEFT_RISK', 'REGION']
    TELEM_FEATURES = GLM_FEATURES + ['telematics_score']

    def _book_lr(r):
        return r['CLAIM_AMOUNT'].sum() / r['FINAL_PREMIUM_SST'].sum() * 100

    def _book_spearman(r):
        p = r.groupby('POLID').agg(c=('CLAIM_COUNT', 'sum'), prem=('FINAL_PREMIUM_SST', 'sum'))
        return spearmanr(p['prem'], p['c']).statistic

    def _retained_lr(r, decline_pct):
        p = r.groupby('POLID').agg(claims=('CLAIM_AMOUNT', 'sum'), prem=('FINAL_PREMIUM_SST', 'sum')).reset_index()
        k = int(np.floor(len(p) * decline_pct))
        p = p.sort_values('prem', ascending=False).iloc[k:]
        return p['claims'].sum() / p['prem'].sum() * 100

    d_tariff = price_book(res, 'tariff', cfg)
    d_glm = price_book(res, 'glm', cfg, train_seed=MODEL_TRAIN_SEED)
    d_telem = price_book(res, 'telem', cfg, train_seed=MODEL_TRAIN_SEED)
    lr_df = pd.DataFrame([{'Regime': 'Tariff', 'Portfolio LR (%)': round(_book_lr(d_tariff), 2),
                           'Prem-Count rho': round(_book_spearman(d_tariff), 4), 'Retained LR 15% (%)': round(_retained_lr(d_tariff, 0.15), 2)},
                          {'Regime': 'GLM', 'Portfolio LR (%)': round(_book_lr(d_glm), 2),
                           'Prem-Count rho': round(_book_spearman(d_glm), 4), 'Retained LR 15% (%)': round(_retained_lr(d_glm, 0.15), 2)},
                          {'Regime': 'GLM+Telematics', 'Portfolio LR (%)': round(_book_lr(d_telem), 2),
                           'Prem-Count rho': round(_book_spearman(d_telem), 4), 'Retained LR 15% (%)': round(_retained_lr(d_telem, 0.15), 2)}])
    lines.append('=== PRICING PROGRESSION (same book re-priced; only pricing differs) ===')
    lines.append(lr_df.to_string(index=False))
    _passed = []

    def _check(name, ok, info=''):
        _passed.append(ok)
        lines.append(f'[{"PASS" if ok else "FAIL"}] {name} {info}')

    cb = d_tariff.loc[d_tariff['CLAIM_OCCURRED'], 'BEHAVIOR_RISK'].mean()
    cn = d_tariff.loc[~d_tariff['CLAIM_OCCURRED'], 'BEHAVIOR_RISK'].mean()
    _check('13a. BEHAVIOR_RISK materially higher for claimants', cb - cn >= 0.005, f'({cb:.3f} vs {cn:.3f}, diff={cb - cn:+.3f})')
    sb = d_tariff.loc[d_tariff['CLAIM_OCCURRED'], 'telematics_score'].mean()
    sn = d_tariff.loc[~d_tariff['CLAIM_OCCURRED'], 'telematics_score'].mean()
    _check('13b. telematics_score lower for claimants', sb < sn, f'({sb:.1f} vs {sn:.1f})')
    _check('13c. portfolio LR: telem < tariff',
           lr_df.loc[2, 'Portfolio LR (%)'] < lr_df.loc[0, 'Portfolio LR (%)'],
           f"({lr_df.loc[2, 'Portfolio LR (%)']:.2f}% < {lr_df.loc[0, 'Portfolio LR (%)']:.2f}%)")
    _check('13d. retained LR: GLM & telem << tariff; telem within 0.5pp of GLM',
           lr_df.loc[1, 'Retained LR 15% (%)'] < lr_df.loc[0, 'Retained LR 15% (%)']
           and lr_df.loc[2, 'Retained LR 15% (%)'] <= lr_df.loc[1, 'Retained LR 15% (%)'] + 0.5,
           f"(tariff {lr_df.loc[0, 'Retained LR 15% (%)']:.2f}% | glm {lr_df.loc[1, 'Retained LR 15% (%)']:.2f}% | telem {lr_df.loc[2, 'Retained LR 15% (%)']:.2f}%)")
    _check('13e. premium-count correlation: telem > glm (prices behavior)',
           lr_df.loc[2, 'Prem-Count rho'] > lr_df.loc[1, 'Prem-Count rho'],
           f"({lr_df.loc[2, 'Prem-Count rho']:.4f} > {lr_df.loc[1, 'Prem-Count rho']:.4f})")

    def _tier_lr(r):
        d = r.copy()
        d['telem_bin'] = pd.cut(d['telematics_score'], bins=[0, 50, 70, 85, 100],
                                labels=['<50 (High Risk)', '50-70 (Moderate)', '70-85 (Low Risk)', '85-100 (Safe)'])
        return d.groupby('telem_bin', observed=False).apply(
            lambda g: g['CLAIM_AMOUNT'].sum() / g['FINAL_PREMIUM_SST'].sum() * 100)

    tier_glm = _tier_lr(d_glm)
    tier_telem = _tier_lr(d_telem)
    _spread_glm = tier_glm.max() - tier_glm.min()
    _spread_telem = tier_telem.max() - tier_telem.min()
    _check('13f. tier LR spread: GLM > telem (blind pricing leaves larger risk gradient)',
           _spread_glm > _spread_telem, f'(GLM {_spread_glm:.1f}pp vs telem {_spread_telem:.1f}pp)')
    lines.append(f'Pricing-progression checks: {sum(_passed)}/{len(_passed)} passed')
    lines += ['', '=== LR by telematics risk tier ===', 'GLM (blind to behavior):', tier_glm.round(1).to_string(),
              'GLM+Telematics (prices behavior):', tier_telem.round(1).to_string()]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    colors = ['#94a3b8', '#f59e0b', '#2563eb']
    for i, (col, ylab, fmt) in enumerate([('Portfolio LR (%)', 'Portfolio Loss Ratio (%)', '.1f'),
                                          ('Retained LR 15% (%)', 'Retained Book LR (%)', '.1f'),
                                          ('Prem-Count rho', 'Spearman rho (premium vs claim count)', '.3f')]):
        axes[i].bar(lr_df['Regime'], lr_df[col], color=colors)
        axes[i].set_ylabel(ylab)
        axes[i].set_title(f'{col} by Pricing Regime')
        for j, v in enumerate(lr_df[col]):
            axes[i].text(j, v + (0.004 if fmt == '.3f' else 0.3), f'{v:{fmt}}', ha='center', fontweight='bold')
    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, 'pricing_progression.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    lines.append('\n[figure] report/figures/pricing_progression.png')
    return '\n'.join(lines), d_tariff, d_glm, d_telem


def _sec_premium_by_tier(d_tariff, d_glm, d_telem):
    lines = []
    t_prem = risk_tier_analysis(d_tariff)['mean_prem'] if d_tariff is not None else None
    g_prem = risk_tier_analysis(d_glm)['mean_prem']
    tm_prem = risk_tier_analysis(d_telem)['mean_prem']
    tiers = list(g_prem.index)
    tbl = pd.DataFrame({'Tariff': t_prem if t_prem is not None else float('nan'),
                        'GLM': g_prem, 'GLM+Telematics': tm_prem}).round(0)
    tbl.index.name = 'Behavior tier'
    lines += ['Mean premium (RM) by telematics behavior tier:', '', tbl.to_string()]
    rel_g = risk_tier_analysis(d_glm).loc['<50 High Risk', 'relativity']
    rel_tm = risk_tier_analysis(d_telem).loc['<50 High Risk', 'relativity']
    ok = rel_tm > rel_g + 0.03
    lines += ['', f'[{"PASS" if ok else "FAIL"}] 13g. high-risk tier pays more under telematics '
                  f'(High/Safe relativity {rel_tm:.2f} > GLM {rel_g:.2f} + 0.03)']
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(tiers)); w = 0.26
    for i, (label, s) in enumerate((('Tariff', t_prem), ('GLM', g_prem), ('GLM+Telematics', tm_prem))):
        if s is not None:
            ax.bar(x + (i - 1) * w, s, w, label=label)
    ax.set_xticks(x); ax.set_xticklabels(tiers)
    ax.set_ylabel('Mean premium (RM)'); ax.set_title('Premium by telematics risk tier')
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, 'premium_by_tier.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    lines.append('\n[figure] report/figures/premium_by_tier.png')
    return '\n'.join(lines)


def generate_actuarial_report(cohort_results, config=None):
    cfg = config or COHORT_CONFIG
    md = ['# Consolidated Actuarial Report']
    md.append(f'\n_Generated {datetime.datetime.now():%Y-%m-%d %H:%M} | '
              f"seed {cfg.get('seed')} | n {cfg.get('n')} | "
              f"expense_loading {cfg.get('expense_loading', 1.40)} | "
              f"telemetric_load {cfg.get('telemetric_load', 1.0)}_")
    md.append('')

    def emit(heading, fn, *args):
        md.append(f'## {heading}'); md.append('')
        try:
            out = fn(*args)
            text = out[0] if isinstance(out, tuple) else out
            extra = out[1:] if isinstance(out, tuple) else ()
            if text:
                md.append('```'); md.append(text); md.append('```'); md.append('')
            return extra
        except Exception as e:
            md.append(f'> _section skipped: {type(e).__name__}: {str(e)[:150]}_'); md.append('')
            return None

    emit('1. Cohort Summary', _sec_cohort_summary, cohort_results)
    emit('2. Validation - Core Tests', _sec_validation_core, cohort_results, cfg)
    emit('3. Validation - Reproducibility + EDA Enrichment', _sec_repro_eda, cohort_results, cfg)
    fd_res = emit('4. Validation - Enhanced Tests', _sec_enhanced_validation, cohort_results, cfg)
    final_dataset = fd_res[0] if fd_res else None
    emit('5. EDA - Policy Trajectories', _sec_eda_trajectories, cohort_results)
    if final_dataset is not None:
        emit('6. EDA - Actuarial Heatmaps', _sec_eda_heatmaps, final_dataset)
    else:
        md.append('## 6. EDA - Actuarial Heatmaps'); md.append(''); md.append('> _skipped (enhanced validation did not run)_'); md.append('')
    emit('7. EV Market Analysis', _sec_ev_analysis, cohort_results, cfg)
    pricing = emit('8. Pricing Progression (Tariff -> GLM -> GLM+Telematics)',
                   _sec_pricing_progression, cohort_results, cfg)
    if pricing:
        _sec_premium_by_tier(*pricing) and emit('9. Risk-Based Pricing - premium by behavior tier',
                                                _sec_premium_by_tier, *pricing)
    else:
        md.append('## 9. Risk-Based Pricing - premium by behavior tier'); md.append('')
        md.append('> _skipped (pricing progression did not run)_'); md.append('')
    md.append('---')
    md.append('## 10. Three-Regime Comparison (Tariff | GLM | GLM+Telematics)')
    md.append('')
    try:
        _cmp = compare_pricing(cohort_results, cfg=cfg, train_seed=MODEL_TRAIN_SEED)
        md.append('```'); md.append(_cmp.to_string(index=False)); md.append('```'); md.append('')
    except Exception as e:
        md.append(f'> _comparison skipped: {type(e).__name__}: {str(e)[:120]}_'); md.append('')
    md.append(f"_Note: expense_loading calibrated to {cfg.get('expense_loading', 1.40):.2f} to target ~70% GLM/telem LR; "
              f"tariff LR is unaffected (tariff formula has no expense loading). "
              f"GLM/telem trained on seed {MODEL_TRAIN_SEED}, tested on seed {cfg.get('seed')}._")
    out = '\n'.join(md)
    with open(os.path.join(REPORT_DIR, 'actuarial_report.md'), 'w', encoding='utf-8') as fh:
        fh.write(out)
    print('=' * 72)
    print('CONSOLIDATED ACTUARIAL REPORT')
    print('=' * 72)
    print(out)
    print(f'\n[report saved] report/actuarial_report.md (+ figures in {FIG_DIR}/)')


try:
    generate_actuarial_report(cohort_results, COHORT_CONFIG)
except NameError:
    print('[report skipped] cohort_results / COHORT_CONFIG not defined - run the simulation first')